## CITF emails analyses

<div style="font-size: 0.85em">

**The main email data** (all loaded together from `/citfdata_share/email/processed data/main_email_dfs_20251007.Rdata`)

| Object | Rows × cols | What it is |
|---|---|---|
| `msgs10` | 34,978 × 32 | **Messages, latest version.** One row per email. Drops the link to the original message ID (`msgid1`), so it has more duplicates, flagged in `del_new`. Includes manual cleaning from `msgr_grp2 (1).xlsx`. This is what the EDA script analyzes. |
| `msgs8` | 39,579 × 31 | **Messages, earlier version.** Keeps `msgid1`, useful for tracing a message back or auditing. |
| `recip4` | 129,654 × 4 | **Recipients, latest version.** One row per recipient per email (to/cc). Drops `msgid1`, so fewer duplicates. Pairs with `msgs10`. |
| `recip3` | 162,511 × 8 | **Recipients, earlier version.** Keeps `msgid1` and extra columns for troubleshooting. Pairs with `msgs8`. |
| `crs` | 403 × 9 | **CITF researchers.** Links individual investigators and staff to their CITF study IDs. People can appear more than once if they're on several studies. |

</div>

##### 1 - Load email data

Loads `msgs10` (messages) and `recip4` (recipients) and checks missingness in each.

In [ ]:
import os
import pandas as pd

os.chdir('/citfdata_share/email/processed data/melissa')

msgs  = pd.read_csv('msgs10.csv')   # 34,978 emails
recip = pd.read_csv('recip4.csv')   # 129,654 recipient rows
counts = {'Start (msgs10)': len(msgs)}

def missingness(df, name):
    out = pd.DataFrame({
        'dtype': df.dtypes,
        'n_missing': df.isna().sum(),
        'pct_missing': (df.isna().mean() * 100).round(1)
    })
    print(f"\n{name}: {df.shape[0]:,} rows × {df.shape[1]} cols")
    display(out)

missingness(msgs, 'msgs10')
missingness(recip, 'recip4')

##### 2 - Flatten emails

Builds a one-row-per-email table (from, to, cc, subject, body) by joining the messages in `msgs10` with their recipients from `recip4`.

In [ ]:
# Drop flagged duplicates (del_new == 1): -2,258 → 32,720 emails
msgs = msgs[msgs['del_new'] != 1]
counts['After del_new filter'] = len(msgs)

# Keep only recipient rows that have an email address: -2,724 → 126,930 rows
recip = recip.dropna(subset=['recipient_email_clean'])

# One row per email: join all 'to' and all 'cc' email addresses into a single string
to = recip[recip['recipient_type'] == 'to'].groupby('msgid2')['recipient_email_clean'].agg('; '.join).rename('to')
cc = recip[recip['recipient_type'] == 'cc'].groupby('msgid2')['recipient_email_clean'].agg('; '.join).rename('cc')

# Left joins, no rows lost: 32,720 emails
emails = (msgs[['msgid2', 'msg_citfid', 'sender_email', 'msg_subject_clean', 'msg_body']]
          .rename(columns={'sender_email': 'from',
                           'msg_subject_clean': 'subject',
                           'msg_body': 'body'})
          .merge(to, on='msgid2', how='left')
          .merge(cc, on='msgid2', how='left')
          [['msgid2', 'msg_citfid', 'from', 'to', 'cc', 'subject', 'body']])
counts['After merges'] = len(emails)

# Drop emails with identical content under different IDs: -3,240 → 29,480 emails
content_cols = ['from', 'to', 'cc', 'subject', 'body']
emails = emails.drop_duplicates(subset=content_cols, keep='first')
counts['After content dedup'] = len(emails)

# Keep only emails with a sender, at least one recipient, and a body: -3,260 → 26,220 emails
emails = emails.dropna(subset=['from', 'to', 'body'])
counts['After from/to/body filter'] = len(emails)

# Summary
summary = pd.DataFrame({'emails': counts})
summary['removed'] = summary['emails'].shift() - summary['emails']
print(summary)

# Columns: N non-missing (% missing)
for col in emails.columns:
    n = emails[col].notna().sum()
    pct = emails[col].isna().mean() * 100
    print(f"{col:<10} {n:>6,} ({pct:.1f}% missing)")

display(emails.head())

##### 3 - Restrict to emails linked to a study

Keeps only emails with a valid CITF study ID in `msg_citfid` (such as `103-CC`), excluding `200-NF`.

In [ ]:
# Keep only emails with a valid study ID (e.g. 103-CC), excluding 200-NF: -15,823 → 10,397 emails
valid = (emails['msg_citfid'].astype(str).str.match(r'^\d{3}-[A-Za-z]{2}$')
         & (emails['msg_citfid'].astype(str).str.upper() != '200-NF'))
emails_study = emails[valid]
counts['After study ID filter'] = len(emails_study)

# Summary
summary = pd.DataFrame({'emails': counts})
summary['removed'] = summary['emails'].shift() - summary['emails']
print(summary)

print("\nDistinct study IDs:", emails_study['msg_citfid'].nunique())

# Columns: N non-missing (% missing)
for col in emails_study.columns:
    n = emails_study[col].notna().sum()
    pct = emails_study[col].isna().mean() * 100
    print(f"{col:<10} {n:>6,} ({pct:.1f}% missing)")

display(emails_study.head())

##### 4 - Install DocuScope

Installs pandas, polars, pyarrow and docuscospacy, and the `en_docusco_spacy` model (version 1.5). Block 5 needs pyarrow to convert polars output to pandas.

In [ ]:
%pip install pandas polars pyarrow docuscospacy
%pip install "https://huggingface.co/browndw/en_docusco_spacy/resolve/main/en_docusco_spacy-1.5-py3-none-any.whl"

##### 5 - Tag email bodies with DocuScope

Loads the `en_docusco_spacy` model and parses the first 100 study-linked emails (`emails_study`) as a test, printing how long a full run of all 10,397 would take. Builds a table of DocuScope tag counts per email (one row per `msgid2`) and converts the counts to proportions. Counts are tagged tokens, including Untagged; block 8 counts tag spans instead. Remove `.head(100)` to parse all 10,397 emails.

In [ ]:
import time
import spacy
import polars as pl
import docuscospacy as ds

nlp = spacy.load('en_docusco_spacy')

# docuscope_parse needs a polars DataFrame with 'doc_id' and 'text' columns
corp = pl.DataFrame({
    'doc_id': emails_study['msgid2'].astype(str).tolist(),
    'text':   emails_study['body'].astype(str).tolist()
})

# Test on a small sample first, then remove .head(100) to run all 10,397
start = time.time()
tokens = ds.docuscope_parse(corp.head(100), nlp_model=nlp, n_process=1, batch_size=25)
elapsed = time.time() - start
print(f"Parsed 100 emails in {elapsed:.0f}s → full run ≈ {elapsed * 104 / 60:.0f} min")

# Document-term matrix of DocuScope tag counts (one row per email)
ds_counts = ds.tags_dtm(tokens, count_by='ds').to_pandas().set_index('doc_id')

# Convert counts to proportions so long and short emails are comparable
ds_props = ds_counts.div(ds_counts.sum(axis=1), axis=0)

print(ds_props.shape)
display(ds_props.head())

##### 6 - View DocuScope tags in an email

Highlights DocuScope categories in a single email body. Call `show_email(msgid2)` with any email ID.

In [ ]:
import itertools
from spacy import displacy

palette = ['#f8d7e3', '#d6e6fb', '#fde2c4', '#e5dcf7', '#d5f0dc', '#fdf3c4']
labels = nlp.get_pipe('ner').labels
colors = dict(zip(labels, itertools.cycle(palette)))

def show_email(msgid2):
    """Show one email's body with DocuScope categories highlighted."""
    body = emails_study.loc[emails_study['msgid2'] == msgid2, 'body'].iloc[0]
    displacy.render(nlp(body), style='ent', jupyter=True, options={'colors': colors})

# Example: the first study email
show_email(emails_study['msgid2'].iloc[0])

##### 7 - Export 30 tagged emails to HTML

Draws 30 study emails at random from those of 300 to 2,000 characters with no credential-like text, highlights their DocuScope categories, and saves them as one HTML page, `docuscope_sample_30.html`, in `docuscope-assessment/data`. Blocks 8 to 12 use the same pool of eligible emails.

In [ ]:
import html
from collections import Counter

# Exclude emails that look like they contain credentials
cred_pattern = (r'password|passwd|\bpwd\b|\bpass\s*:|mot de passe|\bmdp\b|username|\bUN\s*:'
                r'|token=|privateurl|access_token|\bsig=|api[_ ]?key')
has_creds = emails_study['body'].str.contains(cred_pattern, case=False, regex=True, na=False)
print("Emails excluded for possible credentials:", has_creds.sum())

# Keep emails of a readable length (characters)
min_chars, max_chars = 300, 2000
body_len = emails_study['body'].str.len()
in_range = body_len.between(min_chars, max_chars)
print(body_len.describe())
print(f"Emails between {min_chars} and {max_chars} chars (no credentials):",
      (in_range & ~has_creds).sum())

# Pick 30 random study emails from the eligible pool
sample = emails_study[~has_creds & in_range].sample(30, random_state=42)

# HOW categories, in a provisional working grouping (not from Brown and Laudenbach 2022)
how_groups = {
    "Interaction": ["Interactive", "Facilitate", "MetadiscourseInteractive", "Inquiry"],
    "Stance":      ["ConfidenceHigh", "ConfidenceHedged", "ForceStressed",
                    "Uncertainty", "Contingent", "Responsibility"],
    "Evaluation":  ["Positive", "Negative"],
    "Planning":    ["Future", "Updates", "Narrative", "Strategic"],
    "Self":        ["FirstPerson"],
    "Cohesion":    ["MetadiscourseCohesive", "Reasoning"],
}

# WHAT: content-bearing tags
what_groups = {
    "Topics":      ["InformationTopics", "AcademicTerms", "PublicTerms"],
    "Reporting":   ["InformationReportVerbs", "InformationExposition", "InformationChange"],
    "Places":      ["InformationPlace"],
    "Description": ["Description"],
}

how_cats  = {c for cats in how_groups.values() for c in cats}
what_cats = {c for cats in what_groups.values() for c in cats}

def phrases_by_cat(doc):
    """Map each tag to its distinct phrases (in order of appearance) and total count."""
    out = {}
    for ent in doc.ents:
        p = " ".join(ent.text.split())
        phrases, n = out.get(ent.label_, ([], 0))
        if p.lower() not in [x.lower() for x in phrases]:
            phrases.append(p)
        out[ent.label_] = (phrases, n + 1)
    return out

def group_block(found, groups):
    """HTML lines: one per group, listing each category with count and phrases."""
    lines = []
    for gname, cats in groups.items():
        items = []
        for cat in cats:
            if cat in found:
                phrases, n = found[cat]
                items.append(f"<i>{cat}</i> ({n}): " + html.escape("; ".join(phrases)))
        if items:
            lines.append(f"<b>{gname}</b> — " + " &nbsp;|&nbsp; ".join(items))
    return "<br>".join(lines) if lines else "<i>none</i>"

# Build each email and tally tags across the sample
totals = Counter()
parts = []
for i, row in enumerate(sample.itertuples(), 1):
    doc = nlp(row.body)
    totals.update(ent.label_ for ent in doc.ents)
    found = phrases_by_cat(doc)
    tagged = displacy.render(doc, style='ent', jupyter=False, page=False,
                             options={'colors': colors})
    parts.append(f"""
    <div class="email">
      <h2>Email {i} of 30</h2>
      <p class="meta">
        <b>msgid2:</b> {row.msgid2} &nbsp;|&nbsp; <b>Study:</b> {html.escape(str(row.msg_citfid))}<br>
        <b>From:</b> {html.escape(str(row._3))}<br>
        <b>Subject:</b> {html.escape(str(row.subject))}<br>
        <b>Length:</b> {len(row.body)} characters
      </p>
      <div class="box how"><div class="label">HOW</div>{group_block(found, how_groups)}</div>
      <div class="box what"><div class="label">WHAT</div>{group_block(found, what_groups)}</div>
      {tagged}
    </div>""")

# ---- Overall summary (top of page) ----
n_all   = sum(totals.values())
n_how   = sum(v for k, v in totals.items() if k in how_cats)
n_what  = sum(v for k, v in totals.items() if k in what_cats)
n_other = n_all - n_how - n_what
pct = lambda n: 100 * n / n_all if n_all else 0

def group_rows(groups, cls):
    rows = []
    for gname, cats in groups.items():
        g_n = sum(totals.get(c, 0) for c in cats)
        detail = ", ".join(f"{c} {totals.get(c, 0)}" for c in cats if totals.get(c, 0))
        rows.append(f"<tr class='{cls}'><td>{gname}</td><td>{g_n}</td>"
                    f"<td>{pct(g_n):.1f}%</td><td>{detail}</td></tr>")
    return "".join(rows)

other_detail = ", ".join(f"{k} {v}" for k, v in totals.most_common()
                         if k not in how_cats and k not in what_cats)

summary = f"""
<div class="summary">
  <h2>Overall tag split (n = 30 emails, {n_all} tags)</h2>
  <div class="bar">
    <div class="seg how-bg"   style="width:{pct(n_how):.1f}%">HOW {pct(n_how):.0f}%</div>
    <div class="seg what-bg"  style="width:{pct(n_what):.1f}%">WHAT {pct(n_what):.0f}%</div>
    <div class="seg other-bg" style="width:{pct(n_other):.1f}%">Other {pct(n_other):.0f}%</div>
  </div>
  <table>
    <tr><th>Group</th><th>Tags</th><th>% of all</th><th>Categories</th></tr>
    <tr class="how head"><td><b>HOW total</b></td><td><b>{n_how}</b></td><td><b>{pct(n_how):.1f}%</b></td><td></td></tr>
    {group_rows(how_groups, 'how')}
    <tr class="what head"><td><b>WHAT total</b></td><td><b>{n_what}</b></td><td><b>{pct(n_what):.1f}%</b></td><td></td></tr>
    {group_rows(what_groups, 'what')}
    <tr class="other head"><td><b>Other</b></td><td><b>{n_other}</b></td><td><b>{pct(n_other):.1f}%</b></td><td>{other_detail}</td></tr>
  </table>
</div>"""

print(f"HOW {n_how} ({pct(n_how):.1f}%) | WHAT {n_what} ({pct(n_what):.1f}%) | Other {n_other} ({pct(n_other):.1f}%)")

page = f"""<!doctype html>
<html><head><meta charset="utf-8"><title>DocuScope sample</title>
<style>
  body {{ font-family: sans-serif; max-width: 900px; margin: 40px auto; padding: 0 16px; }}
  .email {{ border-top: 2px solid #ccc; padding: 24px 0; }}
  .meta {{ color: #555; font-size: 0.9em; }}
  .box {{ padding: 8px 12px; margin: 8px 0; font-size: 0.85em; line-height: 1.6; border-left: 4px solid; }}
  .box .label {{ font-weight: bold; letter-spacing: 0.05em; margin-bottom: 4px; }}
  .how  {{ background: #f3f0fa; border-color: #8a75c9; }}
  .what {{ background: #eef6f1; border-color: #5a9e74; }}
  .summary {{ margin: 24px 0 32px; }}
  .bar {{ display: flex; height: 32px; border-radius: 6px; overflow: hidden; margin: 12px 0; }}
  .seg {{ color: #fff; font-size: 0.85em; font-weight: bold; display: flex;
          align-items: center; justify-content: center; white-space: nowrap; overflow: hidden; }}
  .how-bg {{ background: #8a75c9; }}  .what-bg {{ background: #5a9e74; }}  .other-bg {{ background: #aaa; }}
  table {{ border-collapse: collapse; width: 100%; font-size: 0.85em; }}
  th, td {{ text-align: left; padding: 5px 8px; border-bottom: 1px solid #e5e5e5; vertical-align: top; }}
  tr.how.head {{ background: #e6e0f5; }}  tr.what.head {{ background: #dcefe3; }}  tr.other.head {{ background: #eee; }}
</style></head>
<body><h1>DocuScope tags: 30 sampled study emails</h1>
<p class="meta">Sample: emails of {min_chars}–{max_chars} characters, credential-like emails excluded,
random_state=42.<br>
HOW = stance, interaction and planning categories, in a provisional working grouping.
WHAT = content-bearing categories. Other = tags in neither list. Counts are tag occurrences.</p>
{summary}
{''.join(parts)}
</body></html>"""

out_path = '/home/zack/CITF-emails/docuscope-assessment/data/docuscope_sample_30.html'
with open(out_path, 'w', encoding='utf-8') as f:
    f.write(page)
print("Saved to:", out_path)

##### 8 - Distribution of HOW tags

Parses all emails eligible for the block 7 sample (300 to 2,000 characters, no credentials), keeps only the HOW categories, and summarizes their distribution by category, email and sender. Counts are tag spans.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

# Same emails as block 7: 300 to 2,000 characters, no credentials
pool = emails_study[~has_creds & in_range]
print(f"Emails in pool: {len(pool):,}")

# Parse all emails in the pool, one row per tagged phrase
start = time.time()
rows = []
for msgid2, doc in zip(pool['msgid2'], nlp.pipe(pool['body'].astype(str), batch_size=50)):
    for ent in doc.ents:
        rows.append((msgid2, ent.label_, " ".join(ent.text.split())))
spans = pd.DataFrame(rows, columns=['msgid2', 'tag', 'phrase'])
print(f"Parsed in {(time.time() - start) / 60:.1f} min: {len(spans):,} tags")

# Keep HOW categories only
how = spans[spans['tag'].isin(how_cats)].copy()
print(f"HOW tags: {len(how):,} of {len(spans):,} ({100 * len(how) / len(spans):.1f}%)")

# HOW totals per category, as % of HOW tags and % of all tags
cat_tab = how['tag'].value_counts().rename('tags').to_frame()
cat_tab['pct_of_how'] = 100 * cat_tab['tags'] / len(how)
cat_tab['pct_of_all'] = 100 * cat_tab['tags'] / len(spans)
display(cat_tab.round(1))
cat_order = list(cat_tab.index)

# HOW tags per email, and HOW share of each email's tags (emails with no tags kept as 0)
per_email = pd.DataFrame({
    'all_tags': spans.groupby('msgid2').size(),
    'how_tags': how.groupby('msgid2').size()
}).reindex(pool['msgid2']).fillna(0)
per_email['how_share'] = per_email['how_tags'] / per_email['all_tags'].replace(0, np.nan)
print("Emails with no HOW tags:", int((per_email['how_tags'] == 0).sum()))
display(per_email[['how_tags', 'how_share']].describe().round(2))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
per_email['how_tags'].plot.hist(bins=40, ax=axes[0])
axes[0].set_xlabel('HOW tags per email')
per_email['how_share'].dropna().plot.hist(bins=40, ax=axes[1])
axes[1].set_xlabel('HOW share of all tags per email')
plt.tight_layout()
plt.show()

# Mix of categories within each email (proportions of the email's HOW tags)
cat_props = (how.groupby(['msgid2', 'tag']).size().unstack(fill_value=0)
             .reindex(columns=cat_order, fill_value=0))
cat_props = cat_props.div(cat_props.sum(axis=1), axis=0)
ax = cat_props.plot.box(figsize=(12, 4), rot=90)
ax.set_ylabel("Share of the email's HOW tags")
plt.show()
display(cat_props.describe().round(2))

# Mix of categories for the 10 most frequent senders
sender = pool.set_index('msgid2')['from']
sender_counts = sender.value_counts()
how['sender'] = how['msgid2'].map(sender)
by_sender = (how[how['sender'].isin(sender_counts.head(10).index)]
             .groupby(['sender', 'tag']).size().unstack(fill_value=0)
             .reindex(columns=cat_order, fill_value=0))
by_sender = by_sender.div(by_sender.sum(axis=1), axis=0)
by_sender.insert(0, 'emails', sender_counts.reindex(by_sender.index))
display(by_sender.sort_values('emails', ascending=False).round(2))

# 15 most frequent phrases per HOW category (lowercased)
how['phrase_lc'] = how['phrase'].str.lower()
top_phrases = (how.groupby(['tag', 'phrase_lc']).size().rename('n').reset_index()
               .sort_values(['tag', 'n'], ascending=[True, False])
               .groupby('tag').head(15))
for cat in cat_order:
    sub = top_phrases[top_phrases['tag'] == cat]
    print(f"\n{cat} ({int((how['tag'] == cat).sum()):,} tags)")
    print("  " + "; ".join(f"{p} ({n})" for p, n in zip(sub['phrase_lc'], sub['n'])))

##### 9 - Label email lines with Chipmunk

Labels every line of every pool email with one of the 15 line labels of Chipmunk (Bevendorff et al. 2020), the email segmenter released with its trained model. DocuScope tags each email's full text as before; the labels only mark which lines are salutation, paragraph, closing, signature, banner (`mua_signature`), quoted text and so on, so that later blocks can place each tag. Chipmunk runs in its own environment (`~/models/email-zoning/chipmunk-env`, TensorFlow 2.2) at the repository commit and package versions recorded beside it, and its output is saved to `email_zones.parquet`. Two labels are kept per line: `chipmunk_raw`, the model's top label, and `chipmunk`, the label after the repository's own correction rules, which the authors applied to build their corpus. Block 10 maps these labels to zones.

In [ ]:
import json, os, subprocess, tempfile, time
import pandas as pd

# Pool from block 7: study emails of 300 to 2,000 characters, no credentials
pool_z = emails_study[~has_creds & in_range][['msgid2', 'body']].copy()
pool_z['msgid2'] = pool_z['msgid2'].astype(str)
bodies = dict(zip(pool_z['msgid2'], pool_z['body'].astype(str)))

zoning_dir = '/home/zack/models/email-zoning'
zack_dir = '/home/zack/CITF-emails/docuscope-assessment/data'
zones_path = os.path.join(zack_dir, 'email_zones.parquet')

# Chipmunk prediction, as run on the five emails, reading and writing one email per JSON line
chipmunk_script = r'''
import json, sys
import numpy as np
from tensorflow.keras import models
from util.mail_classification import load_fasttext_model, MailLinesSequence
from parsing.message_segmenter import CONTEXT_SHAPE, INF_BATCH_SIZE, _post_process_labels

load_fasttext_model("weights/fasttext-model.bin")
model = models.load_model("weights/segmenter_hinge.epoch-11.val_loss-0.093.h5")
with open(sys.argv[1]) as fin, open(sys.argv[2], "w") as fout:
    for row in fin:
        email = json.loads(row)
        text = email["body"].rstrip("\n")
        try:
            seq = MailLinesSequence(text, CONTEXT_SHAPE, labeled=False, input_is_raw_text=True, batch_size=INF_BATCH_SIZE)
            soft = model.predict(seq)
            raw = [MailLinesSequence.LABEL_MAP_INVERSE[int(np.argmax(s))] for s in soft]
            post = [label for _, label in _post_process_labels(seq, soft.copy())]
            fout.write(json.dumps({"msgid2": email["msgid2"], "raw": raw, "zone": post}) + "\n")
        except Exception as e:
            fout.write(json.dumps({"msgid2": email["msgid2"], "error": repr(e)}) + "\n")
'''

# Email text stays on the CITF storage: the temporary folder sits in docuscope-assessment/data
start_time = time.time()
with tempfile.TemporaryDirectory(dir=zack_dir) as tmp:
    in_path, out_path_z, script_path = (os.path.join(tmp, f) for f in ('in.jsonl', 'out.jsonl', 'chipmunk_predict.py'))
    with open(in_path, 'w', encoding='utf-8') as f:
        for m, b in bodies.items():
            f.write(json.dumps({'msgid2': m, 'body': b}, ensure_ascii=False) + '\n')
    with open(script_path, 'w') as f:
        f.write(chipmunk_script)
    env = dict(os.environ, CUDA_VISIBLE_DEVICES='', TF_CPP_MIN_LOG_LEVEL='3', PYTHONWARNINGS='ignore',
               PYTHONPATH=os.path.join(zoning_dir, 'acl20-crawling-mailing-lists/src'))
    run = subprocess.run([os.path.join(zoning_dir, 'chipmunk-env/bin/python'), script_path, in_path, out_path_z],
                         cwd=zoning_dir, env=env, capture_output=True, text=True, timeout=4 * 3600)
    if run.returncode != 0:
        print(run.stderr[-3000:])
        raise RuntimeError('Chipmunk failed')
    results = [json.loads(l) for l in open(out_path_z, encoding='utf-8')]

# One row per line, with character offsets into the body DocuScope tagged
rows, errors, mismatches = [], [], []
for rec in results:
    if 'error' in rec:
        errors.append(rec)
        continue
    lines = bodies[rec['msgid2']].rstrip('\n').split('\n')
    if len(lines) != len(rec['zone']):
        mismatches.append(rec['msgid2'])
        continue
    pos = 0
    for i, (line, raw, zone) in enumerate(zip(lines, rec['raw'], rec['zone']), 1):
        rows.append({'msgid2': rec['msgid2'], 'line': i, 'start': pos, 'end': pos + len(line),
                     'text': line, 'chipmunk_raw': raw, 'chipmunk': zone})
        pos += len(line) + 1

zones = pd.DataFrame(rows)
zones.to_parquet(zones_path, index=False)

print(f"Chipmunk run time: {(time.time() - start_time) / 60:.1f} minutes")
print("Pool emails:", len(bodies))
print("Emails zoned:", zones['msgid2'].nunique())
print("Emails with errors:", len(errors))
print("Emails where line counts differ:", len(mismatches))
print("Lines:", len(zones))
print()
print(zones['chipmunk'].value_counts().to_string())
print()
print("Saved to:", zones_path)

##### 10 - Refine email zones

Maps Chipmunk's labels to six zones (authored, signature, banner, quoted, markup, other) and compares them line by line with zones assigned by regular expressions. Then refines the zones with five rules:

1. Closing lines that Chipmunk labels signature, banner or other are authored.
2. Authored, other or signature lines with disclaimer wording are banner.
3. Authored lines that do not end a sentence, between signature lines, are signature.
4. Banner or other lines that read as prose, between authored lines, are authored.
5. Short lines without sentence punctuation at the end of the message are signature when a signature line follows them or they hold a name, title, organization or contact detail. A one-word sign-off name stays authored, and so does a two-word name under a closing with no credentials and nothing of the signature after it.

Zones from regular expressions then take precedence, in this order: markup, attachment and meeting-invitation lines (other), banner, and quoted, which runs from the first reply or forward header to the end of the email. Saves each line's zone and the rule that set it to `email_zones.parquet`.

In [ ]:
import re
from collections import Counter
import pandas as pd

def show(df):
    print(df.to_string(max_colwidth=90))

zones = pd.read_parquet(zones_path)

# ---- Pool: block 7 emails, minus automated senders ----
automated_senders = [r"docusign", r"no-?reply", r"mailer-daemon", r"postmaster", r"microsoft"]
pool10 = emails_study[~has_creds & in_range][['msgid2', 'from', 'body']].copy()
pool10['msgid2'] = pool10['msgid2'].astype(str)
pool10 = pool10[~pool10['from'].astype(str).str.contains('|'.join(automated_senders), case=False, regex=True)]

# ---- Zone patterns, with the header and banner forms the comparison found missing ----
quote_patterns = [
    r"\b(on|le)\s[^\n]{4,200}?(\n[^\n]{0,200}?)?\s(wrote|a écrit)\s*:",
    r"\bwrote on [^\n]{4,60}:",
    r"^\s*-{2,}\s*(original message|original appointment|message d['’]origine|forwarded message|message transféré)",
    r"^\s*begin forwarded message\s*:",
    r"^\s*\*?(from|de)\*?\s*:[^\n]*\n(?:[^\n]*\n){0,3}?\s*\*?(sent|date|envoyé)\*?\s*:",
    r"^\s*_{10,}\s*\n(?:[^\n]*\n){0,3}?\s*\*?(from|de)\*?\s*:",
]
banner_patterns = [
    r"caution\s*!?\s*this (message|email) was sent from outside",
    r"\[?caution\s*:\s*(non-[\w-]+ email|external (mail|email))",
    r"caution\s*:\s*external sender",
    r"some people who received this message don['’]?t often get email",
    r"you don['’]?t often get email from",
    r"learn why this is important",
    r"(avertissement automatis|courriel externe|faites preuve de prudence|ne cliquez (pas|sur))",
    r"exercise caution when opening",
    r"do not click (on )?links",
    r"this (message|email) (originated|came) from outside",
    r"work from home alert",
    r"(located on|acknowledge|homeland)[^\n]{0,200}(traditional|ancestral|unceded|treaty|original)[^\n]{0,80}(territor|lands?\b)",
    r"exp[ée]diteur externe",
    r"certaines personnes qui ont re[çc]u ce (courrier|message)",
    r"^\s*notice\s*:\s*the information (transmitted|contained)",
    r"^\s*confidential(ity)? (warning|notice)\s*:?\s*$",
    r"confidentialit[ée]\s*/\s*confidentiality",
    r"^\W*working remotely during covid",
]
closing = (r"(best|best regards|kind regards|warm regards|regards|cheers|all the best|best wishes|"
           r"sincerely|merci|cordialement|(many )?(thanks|thank you)( so much| again)?"
           r"( for your (time|help|patience))?)")
signoff_pattern = re.compile(
    rf"^\s*((thanks|thank you)( again)?,? and )?{closing}\s*[,.!]*\s*([A-Z][\w'’.-]*(\s[A-Z][\w'’.-]*){{0,2}})?\s*$",
    re.I)
markup_pattern = re.compile(
    r"<[^<>\n]*>[>\]]*|\[?cid\s*:[^\s\]]*\]?|(?:https?|imap|file)://\S+|\[image[^\]]*\]|image removed by sender\.?"
    r"|microsoft outlook web access\s*:?|\bsignature_\d+\b|\b[\w.-]+\.(?:png|jpe?g|gif)\b"
    r"|^\s*(?:[\w&'’.,-]+\s+){0,7}logo\b(?:\s*description automatically generated[^\n]*)?\s*$"
    r"|^\s*[\w.&-]*\s*e-?mail signature\s*$", re.I)
# Attachment lists and meeting-invitation lines (zone other)
attach_pattern = re.compile(
    r"\t\s*[\d.,]+\s*(?:bytes|[KMG]B)\s*$|^\s*attachments\s*:\s*$"
    r"|^\s*<[^<>\n]+\.(?:pdf|docx?|xlsx?|pptx?|csv|zip|txt)>\s*$"
    r"|^\s*(?:microsoft teams meeting|join on your computer[^\n]*|click here to join the meeting|or call in \(audio only\)"
    r"|phone conference id\s*:[^\n]*|\+?[\d ().-]{7,},,\d+#[^\n]*|meeting id\s*:[^\n]*|passcode\s*:[^\n]*|find a local number[^\n]*"
    r"|learn more \| meeting options|join with a video conferencing device|download teams \| join on the web)\s*$", re.I)
template_min_chars, template_min_emails = 30, 20
disclaimer_pattern = re.compile(
    r"confidential|privileged|intended recipient|destinataire|unauthori[sz]ed|prohibited", re.I)

def reused_sentence(line):
    return (len(line.split()) >= 6 and line.rstrip()[-1:] in '.?!'
            and not disclaimer_pattern.search(line))

def first_name(address):
    local = str(address).split('@')[0].lower()
    parts = re.split(r'[._-]', local)
    name = re.sub(r'\d', '', parts[0])
    return name.capitalize() if len(parts) > 1 and len(name) > 2 else None

def line_spans(text):
    pos = 0
    for line in text.split('\n'):
        yield pos, pos + len(line), line
        pos += len(line) + 1

def quoted_start(body):
    starts = [m.start() for p in quote_patterns for m in re.finditer(p, body, re.I | re.M)]
    return min(starts) if starts else None

def signature_start(text, sender):
    end = None
    for s, e, line in line_spans(text):
        if signoff_pattern.match(line):
            end = e
    if end is not None:
        return end
    name = first_name(sender)
    if name:
        hits = [m.start(2) for m in re.finditer(rf"(^|[.!?,]\s+)({name})\b", text, re.M)
                if m.start(2) > len(text) / 2]
        if hits:
            return hits[-1]
    return None

def unquoted(body):
    q = quoted_start(body)
    return body if q is None else body[:q]

line_counts = Counter()
for body in pool10['body'].astype(str):
    line_counts.update({l.strip() for l in unquoted(body).split('\n') if len(l.strip()) >= template_min_chars})
template_lines = {l for l, n in line_counts.items()
                  if n >= template_min_emails and not reused_sentence(l)}

region_order = ['quoted', 'banner', 'signature', 'template']

def regions(body, sender):
    q = quoted_start(body)
    own = body if q is None else body[:q]
    out = []
    if q is not None:
        out.append((q, len(body), 'quoted'))
    for s, e, line in line_spans(own):
        for p in banner_patterns:
            m = re.search(p, line, re.I)
            if m:
                close = line.find(']', m.start()) if line[m.start()] == '[' else -1
                end = s + close + 1 if close >= 0 else min(e, s + m.start() + 300)
                out.append((s + m.start(), end, 'banner'))
                break
        if line.strip() in template_lines:
            out.append((s, e, 'template'))
    sig = signature_start(own, sender)
    if sig is not None:
        out.append((sig, len(own), 'signature'))
    out.sort(key=lambda r: region_order.index(r[2]))
    return out

# ---- Pattern zone for every Chipmunk line ----
chip_map = {'paragraph': 'authored', 'salutation': 'authored', 'closing': 'authored',
            'personal_signature': 'signature', 'mua_signature': 'banner',
            'quotation': 'quoted', 'quotation_marker': 'quoted', 'inline_headers': 'quoted'}

senders = dict(zip(pool10['msgid2'], pool10['from']))
bodies10 = dict(zip(pool10['msgid2'], pool10['body'].astype(str)))
regs = {m: regions(b, senders[m]) for m, b in bodies10.items()}

def pattern_zone(m, s, e, text):
    if markup_pattern.sub('', text).strip() == '':
        return 'markup'
    return next((lab for rs, re_, lab in regs[m] if s < re_ and rs < e), 'authored')

cmp = zones[zones['msgid2'].isin(bodies10) & (zones['text'].str.strip() != '')].copy()
cmp['patterns'] = [pattern_zone(m, s, e, t) for m, s, e, t in zip(cmp['msgid2'], cmp['start'], cmp['end'], cmp['text'])]
cmp['chipmunk_mapped'] = cmp['chipmunk'].map(chip_map).fillna('other')
cmp['agree'] = cmp['patterns'] == cmp['chipmunk_mapped']

# ---- Comparison report ----
print(f"Emails compared: {cmp['msgid2'].nunique():,} | non-blank lines: {len(cmp):,}")
print(f"Template lines (≥{template_min_chars} characters, in ≥{template_min_emails} emails): {len(template_lines):,}")
print(f"Lines where both methods give the same zone: {cmp['agree'].mean():.3f}")

print("\nAgreement per pattern zone (share of its lines Chipmunk places in the same zone):")
show(cmp.groupby('patterns')['agree'].agg(lines='size', agreement='mean').round(3))

print("\nPattern zone (rows) by Chipmunk zone (columns), line counts:")
show(pd.crosstab(cmp['patterns'], cmp['chipmunk_mapped'], margins=True))

print("\nChipmunk's own zones within each pattern zone:")
show(pd.crosstab(cmp['chipmunk'], cmp['patterns']))

print("\nSampled disagreements (up to 10 random lines per combination):")
dis = cmp[~cmp['agree']]
for (p, c), g in dis.groupby(['patterns', 'chipmunk_mapped']):
    sub = g.sample(min(len(g), 10), random_state=42)
    print(f"\npatterns: {p} | chipmunk: {c} ({len(g):,} lines)")
    show(pd.DataFrame({'msgid2': sub['msgid2'], 'chipmunk': sub['chipmunk'],
                       'text': sub['text'].str.strip().str.slice(0, 90)}))

# ---- Refined zones ----
final = zones[zones['msgid2'].isin(bodies10)].copy()
final['patterns'] = [pattern_zone(m, s, e, t) if t.strip() else 'empty'
                     for m, s, e, t in zip(final['msgid2'], final['start'], final['end'], final['text'])]
final['chipmunk_mapped'] = final['chipmunk'].map(chip_map).fillna('other')
final['zone'] = final['chipmunk_mapped']
final['rule'] = ''
blank = final['text'].str.strip() == ''

# Rule 1: closing lines are authored
is_closing = final['text'].str.match(signoff_pattern) & ~blank
r1 = is_closing & final['zone'].isin(['signature', 'banner', 'other'])
final.loc[r1, ['zone', 'rule']] = ['authored', 'closing line']

# Rule 2: authored, other or signature lines with wording found only in disclaimers are banner
disclaimer_only = re.compile(
    r"intended recipient|intended (?:solely |only )?for the (?:use|person|individual|addressee)|privileged"
    r"|destinataire|received this (?:e-?mail|message|communication) in error", re.I)
r2 = final['zone'].isin(['authored', 'other', 'signature']) & final['text'].str.contains(disclaimer_only) & ~blank
final.loc[r2, ['zone', 'rule']] = ['banner', 'disclaimer wording']

# Nearest non-blank neighbours within the same email, read after rules 1 and 2
nb_lines = final[~blank]
prev_zone = nb_lines.groupby('msgid2')['zone'].shift(1).reindex(final.index)
next_zone = nb_lines.groupby('msgid2')['zone'].shift(-1).reindex(final.index)

# Rule 3: authored lines between signature lines are signature (closing lines excepted)
ends_sentence = final['text'].str.rstrip().str[-1:].isin(['.', '?', '!'])
r3 = ((final['zone'] == 'authored') & (prev_zone == 'signature') & (next_zone == 'signature')
    & ~is_closing & ~ends_sentence & ~blank)

# Rule 4: banner or other lines that read as prose, between authored lines, are authored
prose = (final['text'].str.count(r"[A-Za-zÀ-ÿ]{2,}") >= 4) & ~final['text'].str.contains('\t')
r4 = (final['zone'].isin(['banner', 'other']) & (prev_zone == 'authored') & (next_zone == 'authored')
      & prose & (final['rule'] == '') & ~blank)

final.loc[r3, ['zone', 'rule']] = ['signature', 'inside signature block']
final.loc[r4, ['zone', 'rule']] = ['authored', 'prose inside authored text']

# Rule 5: short lines without sentence punctuation at the end of the authored text are signature.
# A run of such lines qualifies when a signature line follows it, or when it holds a name, title,
# organization or contact detail. A one-word sign-off name stays authored, and so does a two-word name
# under a closing line (or labelled closing by Chipmunk) with no credentials and nothing of the signature after it.
# Lines ending in a colon or comma never count.
sig_like = re.compile(
    r"@|\+?\(?\d{3}\)?[ .-]?\d{3}[ .-]?\d{4}\b|\b(?:ph\.?d|m\.?d|m\.?sc|mph|m\.?eng|b\.?sc|ll\.?[bm]|frcpc|faaa\w*|faap"
    r"|m\.?a|j\.?d|she/her|he/him|they/them|elle|professor|director|manager|coordinator|assistant|associate|scientist"
    r"|officer|head|chair|lead|analyst|university|institute|hospital|task force|centre|center|secretariat)\b", re.I)
words = final['text'].str.split().str.len()
# A full stop ends a sentence unless the line is short and carries credentials ("Oscar Palma, M.A., J.D.")
sentence = (final['text'].str.rstrip().str[-1:].isin(['?', '!'])
            | (ends_sentence & ((words >= 5) | ~final['text'].str.contains(sig_like))))
cand = ((final['zone'] == 'authored') & ~blank & ~is_closing & ~sentence & (words <= 12)
        & (final['chipmunk'] != 'salutation') & ~final['text'].str.rstrip().str[-1:].isin([':', ','])
        & ~final['text'].str.match(r"\s*([-•*–]|\d+[.)])\s"))
r5 = pd.Series(False, index=final.index)
for m, g in final[~blank].groupby('msgid2', sort=False):
    idx, zs, cs = list(g.index), g['zone'].tolist(), cand[g.index].tolist()
    i = 0
    while i < len(idx):
        if not cs[i]:
            i += 1
            continue
        j = i
        while j < len(idx) and cs[j]:
            j += 1
        run, nxt = idx[i:j], (zs[j] if j < len(idx) else None)
        if nxt != 'authored':
            texts = final.loc[run, 'text']
            if nxt == 'signature' or texts.str.contains(sig_like).any():
                first = texts.iloc[0]
                signoff = (i > 0 and is_closing[idx[i - 1]]) or final.at[run[0], 'chipmunk'] == 'closing'
                alone = len(run) == 1 and nxt != 'signature'
                if len(first.split()) <= 1 or (signoff and alone and len(first.split()) <= 2 and not sig_like.search(first)):
                    run = run[1:]
                r5[run] = True
        i = j
final.loc[r5, ['zone', 'rule']] = ['signature', 'signature lines after the message']

# Pattern zones take precedence, in this order: markup, attachment and meeting lines (other), banner, quoted
attach = final['text'].str.contains(attach_pattern) & ~blank
for z, hit, name in [('markup', final['patterns'] == 'markup', 'pattern markup'),
                     ('other', attach, 'attachment or meeting line'),
                     ('banner', final['patterns'] == 'banner', 'pattern banner'),
                     ('quoted', final['patterns'] == 'quoted', 'pattern quoted')]:
    final.loc[hit & (final['zone'] != z), 'rule'] = name
    final.loc[hit, 'zone'] = z
final.loc[blank, ['zone', 'rule']] = ['empty', '']
final.to_parquet(zones_path, index=False)

# ---- Refined-zone report ----
nb = final[final['zone'] != 'empty']
print("\nRefined zones (non-blank lines):")
show(nb['zone'].value_counts().rename('lines').to_frame())

print("\nLines changed from Chipmunk's zone, by rule:")
show(nb[nb['rule'] != ''].groupby('rule').agg(lines=('zone', 'size'),
     changes=('zone', lambda z: '; '.join(f"{k} {v}" for k, v in
              (nb.loc[z.index, 'chipmunk_mapped'] + ' → ' + z).value_counts().items()))))

print("\nSampled lines per rule (up to 10):")
for r, g in nb[nb['rule'].isin(['closing line', 'disclaimer wording', 'inside signature block',
                                'prose inside authored text', 'signature lines after the message',
                                'attachment or meeting line'])].groupby('rule'):
    sub = g.sample(min(len(g), 10), random_state=42)
    print(f"\n{r} ({len(g):,} lines)")
    show(pd.DataFrame({'msgid2': sub['msgid2'], 'chipmunk': sub['chipmunk'], 'zone': sub['zone'],
                       'text': sub['text'].str.strip().str.slice(0, 90)}))

print(f"\nSaved with columns {list(final.columns)} to: {zones_path}")

##### 11 - Draw a sample for checking zones by hand

Draws emails from `email_zones.parquet` for checking zones by hand. With `test_round = False`, it draws 80 emails: 20 at random, then 20 each from emails with banner lines, emails with quoted text, and emails where block 10 changed a Chipmunk zone, each drawn from emails not already in the sample. Rule 5 and the attachment and meeting-invitation lines in block 10 come from these 80 emails. With `test_round = True`, it draws `n_test` emails at random from outside the 80, to measure the accuracy of the zones; `n_test` rises in steps of 20 until the lower end of the 95% interval for precision of the authored zone reaches 95%.

Files take the prefix `zone_check` or `heldout_check`. Each draw is saved to `_emails.csv` and reused, so rerunning block 10 cannot change it. Saves the drawn emails' lines to `_sample.csv` and writes a labelling tool to `_tool.html`. The tool shows each email without Chipmunk's labels or the zones; the coder marks where each zone starts and exports a CSV, saved as `_coder1.csv`. Compares the coder's zones with Chipmunk's and with the zones in `email_zones.parquet`.

In [ ]:
import os
import json
import pandas as pd

# ---- Settings ----
# test_round = False: the 80 emails labelled first, which were then used to refine block 10's rules.
# test_round = True: new random emails that block 10's rules have never seen, for a clean accuracy figure.
test_round = True
n_test = 40            # emails in the test round; raise by 20 if the stopping rule below says so
zack_dir = '/home/zack/CITF-emails/docuscope-assessment/data'
zones_path = os.path.join(zack_dir, 'email_zones.parquet')
prefix = 'heldout_check' if test_round else 'zone_check'
sample_path = os.path.join(zack_dir, f'{prefix}_sample.csv')
tool_path = os.path.join(zack_dir, f'{prefix}_tool.html')
emails_path = os.path.join(zack_dir, f'{prefix}_emails.csv')   # the drawn emails, frozen after the first run
dev_emails_path = os.path.join(zack_dir, 'zone_check_emails.csv')
n_per_stratum = 20
seed = 42

# ---- Emails and what their zones contain ----
z = pd.read_parquet(zones_path)
z = z.sort_values(['msgid2', 'start']).reset_index(drop=True)
z['text'] = z['text'].fillna('').astype(str)
nonblank = z['text'].str.strip() != ''
per_email = (z[nonblank].assign(changed=lambda d: d['zone'] != d['chipmunk_mapped'])
             .groupby('msgid2')
             .agg(banner=('zone', lambda s: (s == 'banner').any()),
                  quoted=('zone', lambda s: (s == 'quoted').any()),
                  changed=('changed', 'any')))

# ---- Draw the sample: random first, then each stratum from emails not yet drawn ----
# The draw depends on block 10's zones, so it is saved once and reused; rerunning block 10 cannot change it.
strata = [('random', per_email.index.to_series()),
          ('banner', per_email.index[per_email['banner']].to_series()),
          ('quoted', per_email.index[per_email['quoted']].to_series()),
          ('changed', per_email.index[per_email['changed']].to_series())]
drawn = {}
for name, pool_ids in strata:
    left = pool_ids[~pool_ids.isin(list(drawn))]
    for m in left.sample(min(len(left), n_per_stratum), random_state=seed):
        drawn[m] = name
coder1_path = os.path.join(zack_dir, 'zone_check_coder1.csv')
if test_round:
    # New random emails, excluding the 80 already labelled; extra draws keep the emails drawn before
    dev_ids = set(pd.read_csv(dev_emails_path, dtype={'msgid2': str})['msgid2'])
    drawn = {}
    if os.path.exists(emails_path):
        saved = pd.read_csv(emails_path, dtype={'msgid2': str})
        drawn = dict(zip(saved['msgid2'], saved['stratum']))
    need = n_test - len(drawn)
    if need > 0:
        pool_ids = pd.Series(sorted(set(per_email.index.astype(str)) - dev_ids - set(drawn)))
        for m in pool_ids.sample(need, random_state=seed + len(drawn)):
            drawn[m] = 'test'
    print(f"Test emails: {len(drawn)} (none of the {len(dev_ids)} emails used to refine block 10)")
elif os.path.exists(emails_path):
    saved = pd.read_csv(emails_path, dtype={'msgid2': str})
    drawn = dict(zip(saved['msgid2'], saved['stratum']))
    print(f"Reusing the saved sample in {emails_path}")
elif os.path.exists(coder1_path):
    # First sample, labelled before the draw was saved: rebuild it from the labels. Its random stratum
    # does not depend on the zones; the other strata were drawn from earlier zones and are marked 'targeted'.
    labelled_ids = pd.read_csv(coder1_path, dtype={'msgid2': str})['msgid2'].unique()
    random_ids = {str(m) for m, s in drawn.items() if s == 'random'}
    drawn = {m: ('random' if m in random_ids else 'targeted') for m in labelled_ids}
    print(f"Rebuilt the sample from the labels in {coder1_path}")
else:
    print("Emails available per stratum:", {name: len(ids) for name, ids in strata})
    drawn = {str(m): s for m, s in drawn.items()}
pd.DataFrame({'msgid2': list(drawn), 'stratum': list(drawn.values())}).to_csv(emails_path, index=False)
print("Emails per stratum:", pd.Series(drawn).value_counts().to_dict())

z['msgid2'] = z['msgid2'].astype(str)
sample = z[z['msgid2'].isin(list(drawn))][['msgid2', 'line', 'start', 'end', 'text']].copy()
sample.insert(0, 'stratum', sample['msgid2'].map(drawn))
sample['manual'] = ''
sample.to_csv(sample_path, index=False)
print(f"Sample: {sample['msgid2'].nunique()} emails, {len(sample):,} lines "
      f"({(sample['text'].str.strip() != '').sum():,} non-blank), saved to: {sample_path}")

# ---- Labelling tool: one offline page; Chipmunk's zones are not included ----
emails = [{'msgid2': str(m), 'lines': g['text'].tolist()}
          for m, g in sample.groupby('msgid2', sort=False)]
order = pd.Series(range(len(emails))).sample(frac=1, random_state=seed).tolist()
emails = [emails[i] for i in order]   # shuffled, so strata are not labelled in blocks

zone_html = r'''<!doctype html>
<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>Zone check</title>
<style>
:root { --bg:#f7f7f5; --card:#fff; --ink:#1f2328; --muted:#6b7075; --line:#e2e2de; --sel:#4b5fc4;
  --authored:#2f8f5b; --signature:#8a5cc4; --banner:#c2453d; --quoted:#b7871a; --markup:#4b8fc4; --other:#7a7f85; }
@media (prefers-color-scheme: dark) { :root { --bg:#17181a; --card:#212327; --ink:#e8e8e6; --muted:#9a9ea3; --line:#33363b; --sel:#8b9cf0; } }
* { box-sizing:border-box; }
body { margin:0; background:var(--bg); color:var(--ink); font:15px/1.45 system-ui, -apple-system, sans-serif; }
header { display:flex; flex-wrap:wrap; gap:10px; align-items:center; padding:10px 16px; border-bottom:1px solid var(--line); background:var(--card); position:sticky; top:0; z-index:2; }
header h1 { font-size:16px; margin:0 8px 0 0; }
select, input[type=text], button { font:inherit; font-size:14px; padding:4px 9px; border:1px solid var(--line); border-radius:6px; background:var(--bg); color:var(--ink); }
button { cursor:pointer; } button.primary { background:var(--sel); color:#fff; border-color:var(--sel); }
.spacer { flex:1; }
.wrap { display:grid; grid-template-columns:minmax(0,1fr) 320px; gap:16px; max-width:1360px; margin:16px auto; padding:0 16px; }
main { min-width:0; }
aside { position:sticky; top:64px; align-self:start; background:var(--card); border:1px solid var(--line); border-radius:10px; padding:12px 14px; font-size:13px; }
aside .z { margin:0 0 10px; }
aside .chip { display:inline-block; padding:1px 8px; border-radius:999px; color:#fff; font-weight:600; margin-bottom:2px; }
aside .keys { border-top:1px solid var(--line); padding-top:8px; color:var(--muted); line-height:1.9; }
@media (max-width:900px) { .wrap { grid-template-columns:1fr; } aside { position:static; } }
.meta { color:var(--muted); font-size:13px; margin-bottom:8px; display:flex; gap:12px; flex-wrap:wrap; }
.email { background:var(--card); border:1px solid var(--line); border-radius:10px; overflow:hidden; }
.row { display:grid; grid-template-columns:86px 1fr; border-top:1px solid var(--line); cursor:pointer; }
.row:first-child { border-top:0; }
.lab { font-size:12px; padding:3px 8px; color:#fff; display:flex; align-items:center; gap:4px; }
.lab.none { color:var(--muted); background:transparent; }
.lab .start { font-weight:700; }
.txt { padding:3px 10px; font-family:ui-monospace, Menlo, monospace; font-size:13px; white-space:pre-wrap; word-break:break-word; min-height:22px; }
.row.blank .txt { color:var(--muted); }
.row.sel { outline:2px solid var(--sel); outline-offset:-2px; }
.note { width:100%; margin-top:10px; }
.nav { display:flex; justify-content:space-between; align-items:center; margin:12px 0; color:var(--muted); }
#about { max-width:1360px; margin:12px auto 0; padding:0 16px; font-size:14px; }
#about .box { background:var(--card); border:1px solid var(--line); border-radius:10px; padding:14px 18px; }
#about[hidden] { display:none; }
#about td { padding:2px 12px 2px 0; vertical-align:top; }
kbd { border:1px solid var(--line); border-bottom-width:2px; border-radius:4px; padding:0 5px; font-size:12px; background:var(--card); }
.empty { text-align:center; color:var(--muted); padding:40px; }
</style></head><body>
<header>
  <h1>Zone check</h1>
  <button id="aboutBtn">About</button>
  <label>Coder <select id="coder"><option value="">choose…</option><option>coder1</option><option>coder2</option></select></label>
  <span class="spacer"></span><span id="count"></span>
  <button id="import">Import</button><input type="file" id="file" accept=".csv" hidden>
  <button id="export" class="primary">Export CSV</button>
</header>
<section id="about" hidden><div class="box">
  <p>Label the zone of every non-blank line in each email. Zones come in runs, so you only mark where each run starts: select the first line of a run and press its number. Every line below takes that zone until the next mark. A single line inside another zone (a URL inside authored text, for example) is its own run: mark it, then mark the line after it.</p>
  <p>The zone definitions and keys are in the panel on the right. A line holding only punctuation (a lone full stop) takes the zone of the run it falls in; do not mark it. Blank lines need no label.</p>
  <p>Label alone, without seeing the other coder's file. Codes save in this browser for each coder; <b>Export CSV</b> writes <code>__PREFIX___coder1.csv</code> or <code>__PREFIX___coder2.csv</code>. Save it in <code>docuscope-assessment/data</code>. <b>Import</b> reloads an exported file. The page contains email text: keep it and the exported files on storage approved for the study data.</p>
</div></section>
<div class="wrap"><main id="main"></main><aside id="side"></aside></div>
<script id="data" type="application/json">__DATA__</script>
<script>
const emails = JSON.parse(document.getElementById('data').textContent);
const ZONES = [
  ['authored', 'Text the sender wrote for this message, including greetings, closings and a name typed as the sign-off (a name alone, or "Best, Ahmed")'],
  ['signature', 'Signature blocks: names, titles, organizations, addresses, contact details. A name followed by a title or contact details starts the signature. The sender\'s signature placed below a quoted message is still signature'],
  ['banner', 'Inserted or standing notices, wherever they appear, including inside a signature: external-sender warnings, confidentiality disclaimers, work-from-home and remote-work notices, land acknowledgements'],
  ['quoted', 'Earlier messages below a reply or forward header, including the header and the separator line (____ or -----Original Message-----) that opens it'],
  ['markup', 'A line that is only markup, also inside a signature: angle-bracket tags, image references, "image removed by sender", URLs, Outlook Web Access link labels. A line mixing an image and text takes the zone of the text'],
  ['other', 'Separators that open no quoted message, tables, attachment lists, meeting-invitation details, code and log lines']];
const $ = id => document.getElementById(id);
$('side').innerHTML = ZONES.map(([z, d], i) => `<div class="z"><span class="chip" style="background:var(--${z})">${i + 1} ${z}</span><div>${d}</div></div>`).join('') +
  '<div class="keys"><kbd>↑</kbd> <kbd>↓</kbd> select line<br><kbd>1</kbd>–<kbd>6</kbd> start a zone at the selected line<br><kbd>0</kbd> remove the mark<br><kbd>←</kbd> <kbd>→</kbd> previous or next email<br><kbd>N</kbd> note</div>';
$('aboutBtn').onclick = () => { $('about').hidden = !$('about').hidden; };
let coder = '', marks = {}, notes = {}, cur = 0, sel = 0;
const key = () => '__PREFIX__-' + coder;
function load() { try { const s = JSON.parse(localStorage.getItem(key())) || {}; marks = s.marks || {}; notes = s.notes || {}; cur = s.cur || 0; } catch (e) { marks = {}; notes = {}; cur = 0; } sel = 0; }
function save() { try { localStorage.setItem(key(), JSON.stringify({marks, notes, cur})); } catch (e) {} }
function esc(s) { return String(s ?? '').replace(/[&<>"]/g, c => ({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;'}[c])); }
function labels(e) {
  const m = marks[e.msgid2] || {}; let z = ''; return e.lines.map((_, i) => (z = m[i] ?? z));
}
function complete(e) { const l = labels(e); return e.lines.every((t, i) => !t.trim() || l[i]); }
function render() {
  const done = emails.filter(complete).length;
  $('count').textContent = `${done} / ${emails.length} emails labelled`;
  if (!coder) { $('main').innerHTML = '<div class="empty">Choose a coder to start. Open About for instructions.</div>'; return; }
  const e = emails[cur], l = labels(e), m = marks[e.msgid2] || {};
  $('main').innerHTML = `
    <div class="meta"><span>Email ${cur + 1} of ${emails.length}</span><span>msgid2 ${esc(e.msgid2)}</span><span>${complete(e) ? 'labelled' : 'not yet labelled'}</span></div>
    <div class="email">${e.lines.map((t, i) => `
      <div class="row ${t.trim() ? '' : 'blank'} ${i === sel ? 'sel' : ''}" data-i="${i}">
        <div class="lab ${l[i] ? '' : 'none'}" style="${l[i] ? `background:var(--${l[i]})` : ''}">${m[i] !== undefined ? '<span class="start">▸</span>' : ''}${l[i] || '—'}</div>
        <div class="txt">${esc(t) || '·'}</div></div>`).join('')}</div>
    <input class="note" id="note" type="text" placeholder="Note on this email (optional)" value="${esc(notes[e.msgid2])}">
    <div class="nav"><button id="prev">← previous</button><button id="next">next →</button></div>`;
  document.querySelectorAll('.row').forEach(r => r.onclick = () => { sel = +r.dataset.i; render(); });
  $('prev').onclick = () => go(-1); $('next').onclick = () => go(1);
  $('note').oninput = ev => { notes[e.msgid2] = ev.target.value; save(); };
  const s = document.querySelector('.row.sel'); if (s) s.scrollIntoView({block: 'nearest'});
}
function go(d) { cur = Math.min(Math.max(cur + d, 0), emails.length - 1); sel = 0; save(); render(); }
function mark(z) { const id = emails[cur].msgid2; marks[id] = marks[id] || {}; if (z) marks[id][sel] = z; else delete marks[id][sel]; save(); render(); }
document.addEventListener('keydown', ev => {
  if (!coder) return;
  if (ev.target.id === 'note') { if (ev.key === 'Escape' || ev.key === 'Enter') ev.target.blur(); return; }
  if (ev.target.tagName === 'SELECT' || ev.target.tagName === 'INPUT') return;
  const n = emails[cur].lines.length;
  if (ev.key === 'ArrowDown') { sel = Math.min(sel + 1, n - 1); ev.preventDefault(); render(); }
  else if (ev.key === 'ArrowUp') { sel = Math.max(sel - 1, 0); ev.preventDefault(); render(); }
  else if (ev.key === 'ArrowRight') go(1);
  else if (ev.key === 'ArrowLeft') go(-1);
  else if (ev.key >= '1' && ev.key <= '6') mark(ZONES[+ev.key - 1][0]);
  else if (ev.key === '0' || ev.key === 'Backspace' || ev.key === 'Delete') mark(null);
  else if (ev.key.toLowerCase() === 'n') { ev.preventDefault(); $('note').focus(); }
});
$('coder').onchange = ev => { coder = ev.target.value; load(); render(); };
function csvCell(v) { const s = String(v ?? ''); return /[",\n]/.test(s) ? '"' + s.replace(/"/g, '""') + '"' : s; }
$('export').onclick = () => {
  if (!coder) { alert('Choose a coder first.'); return; }
  const rows = [['msgid2', 'line_index', 'manual', 'note', 'start_mark'].join(',')];
  emails.forEach(e => { const l = labels(e), m = marks[e.msgid2] || {};
    e.lines.forEach((_, i) => rows.push([e.msgid2, i, l[i] || '', i === 0 ? csvCell(notes[e.msgid2]) : '', m[i] !== undefined ? 1 : 0].join(','))); });
  const a = document.createElement('a');
  a.href = URL.createObjectURL(new Blob([rows.join('\n') + '\n'], {type: 'text/csv'}));
  a.download = `__PREFIX___${coder}.csv`; a.click();
};
function parseCSV(text) {
  const out = []; let row = [], cell = '', q = false;
  for (let i = 0; i < text.length; i++) { const ch = text[i];
    if (q) { if (ch === '"' && text[i + 1] === '"') { cell += '"'; i++; } else if (ch === '"') q = false; else cell += ch; }
    else if (ch === '"') q = true; else if (ch === ',') { row.push(cell); cell = ''; }
    else if (ch === '\n') { row.push(cell); out.push(row); row = []; cell = ''; } else if (ch !== '\r') cell += ch; }
  if (cell || row.length) { row.push(cell); out.push(row); } return out;
}
$('import').onclick = () => { if (!coder) { alert('Choose a coder first.'); return; } $('file').click(); };
$('file').onchange = async ev => {
  const f = ev.target.files[0]; if (!f) return;
  const [h, ...body] = parseCSV(await f.text());
  const iM = h.indexOf('msgid2'), iL = h.indexOf('line_index'), iZ = h.indexOf('manual'), iN = h.indexOf('note'), iS = h.indexOf('start_mark');
  marks = {}; notes = {};
  body.forEach(r => { if (r.length < 3) return; const id = r[iM], i = +r[iL];
    if (r[iS] === '1' && r[iZ]) { marks[id] = marks[id] || {}; marks[id][i] = r[iZ]; }
    if (r[iN]) notes[id] = r[iN]; });
  save(); render(); ev.target.value = '';
};
render();
</script></body></html>
'''
data_json = json.dumps(emails, ensure_ascii=False).replace('</', '<\\/')
with open(tool_path, 'w', encoding='utf-8') as f:
    f.write(zone_html.replace('__PREFIX__', prefix).replace('__DATA__', data_json))
print(f"Labelling tool saved to: {tool_path}")

# ---- Compare manual zones with Chipmunk and block 10 (runs once a coder's export is in zack_dir) ----
def show(df):
    print(df.to_string(max_colwidth=90))

coder_files = {c: os.path.join(zack_dir, f'{prefix}_{c}.csv') for c in ('coder1', 'coder2')}
coder_files = {c: p for c, p in coder_files.items() if os.path.exists(p)}
labelled = {}
if not coder_files:
    print(f"\nNo coder exports yet: label the emails, export, and save {prefix}_coder1.csv in", zack_dir)
else:
    ref = sample.drop(columns='manual').copy()
    ref['msgid2'] = ref['msgid2'].astype(str)
    ref = ref.sort_values(['msgid2', 'start'])
    ref['line_index'] = ref.groupby('msgid2').cumcount()
    zz = z[['msgid2', 'start', 'chipmunk_mapped', 'zone']].copy()
    zz['msgid2'] = zz['msgid2'].astype(str)
    ref = ref.merge(zz, on=['msgid2', 'start'], how='left')
    ref = ref[ref['text'].str.strip() != '']
    for c, p in coder_files.items():
        lab = pd.read_csv(p, dtype={'msgid2': str})
        lab['manual'] = lab['manual'].fillna('')
        d = ref.merge(lab[['msgid2', 'line_index', 'manual']], on=['msgid2', 'line_index'], how='left')
        d['manual'] = d['manual'].fillna('')
        labelled[c] = d
        done = d[d['manual'] != '']
        print(f"\n===== {c}: {done['msgid2'].nunique()} emails, {len(done):,} of {len(d):,} non-blank lines labelled =====")
        if done.empty:
            continue
        done = done.assign(chipmunk_ok=done['chipmunk_mapped'] == done['manual'],
                           final_ok=done['zone'] == done['manual'])
        print(f"Agreement with manual labels: Chipmunk {done['chipmunk_ok'].mean():.3f} | block 10 final {done['final_ok'].mean():.3f}")
        print("\nAgreement per manual zone (share of the zone's lines each method labels the same):")
        show(done.groupby('manual').agg(lines=('manual', 'size'), chipmunk=('chipmunk_ok', 'mean'),
                                        final=('final_ok', 'mean')).round(3))
        print("\nAgreement per stratum:")
        show(done.groupby('stratum').agg(emails=('msgid2', 'nunique'), lines=('manual', 'size'),
                                         chipmunk=('chipmunk_ok', 'mean'), final=('final_ok', 'mean')).round(3))
        print("\nManual zone (rows) by block 10 final zone (columns), line counts:")
        show(pd.crosstab(done['manual'], done['zone'], margins=True))
        print("\nManual zone (rows) by Chipmunk zone (columns), line counts:")
        show(pd.crosstab(done['manual'], done['chipmunk_mapped'], margins=True))
        print("\nLines where block 10's final zone differs from the manual label (up to 10 per combination):")
        for (m, f), g in done[~done['final_ok']].groupby(['manual', 'zone']):
            print(f"\nmanual: {m} | block 10: {f} ({len(g)} lines)")
            show(g.sample(min(len(g), 10), random_state=seed)[['msgid2', 'stratum', 'chipmunk_mapped', 'text']])
    if len(labelled) == 2:
        both = labelled['coder1'].merge(labelled['coder2'][['msgid2', 'line_index', 'manual']],
                                        on=['msgid2', 'line_index'], suffixes=('_1', '_2'))
        both = both[(both['manual_1'] != '') & (both['manual_2'] != '')]
        po = (both['manual_1'] == both['manual_2']).mean()
        p1 = both['manual_1'].value_counts(normalize=True)
        p2 = both['manual_2'].value_counts(normalize=True)
        pe = (p1 * p2.reindex(p1.index).fillna(0)).sum()
        print(f"\nCoder agreement on {len(both):,} lines: {po:.3f} (Cohen's kappa {(po - pe) / (1 - pe):.3f})")

# ---- Test round: precision of block 10's authored zone, and the stopping rule ----
# Stop when the lower end of the 95% interval is at least 95%. The interval is the lower of an exact
# binomial interval on lines and a bootstrap over emails (which allows for errors clustering in an email).
if test_round and 'coder1' in labelled:
    import math
    import numpy as np

    def cp_lower(k, n, alpha=0.025):
        """Exact (Clopper-Pearson) lower bound: the p at which P(X >= k | n, p) = alpha."""
        if k == 0:
            return 0.0
        def tail(p):
            return sum(math.exp(math.lgamma(n + 1) - math.lgamma(i + 1) - math.lgamma(n - i + 1)
                                + i * math.log(p) + (n - i) * math.log1p(-p)) for i in range(k, n + 1))
        lo, hi = 0.0, 1.0
        for _ in range(100):
            mid = (lo + hi) / 2
            lo, hi = (mid, hi) if tail(mid) < alpha else (lo, mid)
        return lo
    t = labelled['coder1']
    t = t[t['manual'] != '']
    a = t[t['zone'] == 'authored']
    k, n = int((a['manual'] == 'authored').sum()), len(a)
    cp_low = cp_lower(k, n)
    per = a.groupby('msgid2')['manual'].agg(lambda s: ((s == 'authored').sum(), len(s)))
    hits = np.array([v[0] for v in per]); tots = np.array([v[1] for v in per])
    rng = np.random.default_rng(seed)
    boots = [hits[i].sum() / tots[i].sum() for i in (rng.integers(0, len(per), len(per)) for _ in range(10000))]
    bs_low = float(np.percentile(boots, 2.5))
    recall_den = int((t['manual'] == 'authored').sum())
    print(f"\nTest round: {t['msgid2'].nunique()} emails, {len(t):,} labelled lines")
    print(f"Authored precision: {k}/{n} = {k / n:.3f} | 95% lower bound: exact {cp_low:.3f}, bootstrap {bs_low:.3f}")
    print(f"Authored recall: {k}/{recall_den} = {k / recall_den:.3f}")
    low = min(cp_low, bs_low)
    print("Stopping rule:", "PASS, the lower bound is at least 95%; stop here." if low >= 0.95
          else f"NOT YET, the lower bound is {low:.3f}; set n_test = {n_test + 20}, rerun, and label the new emails.")

##### 12 - Detect likely tagging errors

Leaves out automated notifications (senders listed in `automated_senders`) and emails without zones, re-parses the remaining block 8 emails, and gives each tag the zone of the line it starts on, from `email_zones.parquet` (block 10). Checks that the lines in the zone file match the email bodies. Flags HOW tags in authored text that match an error rule; each rule names the categories it targets. Rules whose matches count as errors without coding (`fixed_rules`) go straight to block 13; the others get a manual check. Lists capitalized runs of tags as candidate names and adjacent tags from different categories as candidate formulas, for issues that span more than one tag. Draws a stratified coding sample (up to 30 tags per checked rule, 10 per domain noun, and 20 per HOW category among tags that match no rule), saves it to `docuscope_coding_sample.csv`, and writes a coding tool for it to `docuscope_coding_tool.html`. All patterns sit at the top of the cell.

In [ ]:
import os
import re
import json
import numpy as np

# Print tables as text so the full report is one copyable output
def show(df):
    print(df.to_string(max_colwidth=90))

# ---- Settings (edit and rerun; all matching is case-insensitive) ----

# Zones of every line, from block 10
zones_path = '/home/zack/CITF-emails/docuscope-assessment/data/email_zones.parquet'

# Senders whose emails are automated notifications; these emails are left out
automated_senders = [r"docusign", r"no-?reply", r"mailer-daemon", r"postmaster", r"microsoft"]

# Multiword names that DocuScope splits into separate tags
name_patterns = [
    r"COVID-19 Immunity Task Force", r"Immunity Task Force",
    r"Groupe de travail sur l['’]immunit[ée]",
    r"Unity Health( Toronto)?", r"McGill University Health Cent(re|er)",
    r"Alberta Health Services", r"Saskatchewan Health Authority", r"Research Institute",
]

# Fixed formulas: (pattern, categories whose tags inside the formula are flagged)
formula_rules = {
    'dont_hesitate': (r"(do not|don['’]?t) hesitate to", {'Uncertainty', 'Negative', 'ForceStressed'}),
    'let_know':      (r"(let|letting) (me|us) know", {'ConfidenceHigh'}),
    'on_my_end':     (r"on (my|our|your|their) end", {'Narrative'}),
    'terms_cond':    (r"terms and conditions", {'Contingent'}),
    'best_of':       (r"to the best of", {'Positive'}),
    'have_a_day':    (r"have a (great|good|nice|wonderful) (day|weekend|week)", {'Strategic', 'Narrative'}),
}

# Nouns from the data sharing domain (tag text, lowercased)
domain_nouns = {
    'agreement', 'agreements', 'consent', 'cohort', 'cohorts', 'assay', 'assays',
    'immunity', 'health', 'research', 'results', 'analysis', 'analyses', 'summary',
    'harmonize', 'harmonization', 'data preparation', 'collected', 'collect', 'access',
}

# Rules whose matches are treated as errors without coding; block 13 applies them directly,
# so they stay out of the coding sample
fixed_rules = {'date_month', 'time_am_pm', 'name', 'french_line', 'terms_cond'}

# French function words, for flagging French lines
fr_words = {'le', 'la', 'les', 'des', 'du', 'de', 'et', 'vous', 'nous', 'est', 'pour',
            'avec', 'sur', 'dans', 'une', 'au', 'aux', 'ce', 'qui', 'que', 'pas', 'à'}
en_words = {'the', 'and', 'to', 'of', 'you', 'is', 'for', 'we', 'in', 'on', 'with', 'this'}

# ---- Helpers ----

def tok(doc, i):
    return doc[i].text.lower() if 0 <= i < len(doc) else ''

# Month names, full or abbreviated
months = (r"jan(uary)?|feb(ruary)?|mar(ch)?|apr(il)?|may|june?|july?|aug(ust)?|sept?(ember)?"
          r"|oct(ober)?|nov(ember)?|dec(ember)?")

# Rules on the tag and its neighbouring tokens: (test, categories flagged; None = any category)
token_rules = {
    'date_month':    (lambda e: re.fullmatch(months, e.text.lower().rstrip('.'))
                                and (re.match(r'\d', tok(e.doc, e.end)) or re.match(r'\d', tok(e.doc, e.start - 1)))
                                and e.label_ != 'Narrative',
                      None),
    'time_am_pm':    (lambda e: re.fullmatch(r'([\d:.]+\s*)?(a\.?m\.?|p\.?m\.?)', e.text.lower())
                                and (re.search(r'\d', e.text) or re.search(r'\d', tok(e.doc, e.start - 1))),
                      {'FirstPerson'}),
    'negated_modal': (lambda e: e.text.lower().split()[-1] in ('can', 'ca', 'do', 'does', 'did')
                                and tok(e.doc, e.end) in ('not', "n't", 'n’t'),
                      {'ForceStressed', 'ConfidenceHedged'}),
    'polite_would':  (lambda e: re.fullmatch(r'((i|we|you|they|she|he|it) )?would( be)?', e.text.lower()),
                      {'Narrative'}),
    'domain_noun':   (lambda e: e.text.lower() in domain_nouns, None),
}

def line_spans(text):
    pos = 0
    for line in text.split('\n'):
        yield pos, pos + len(line), line
        pos += len(line) + 1

def french_lines(text):
    out = []
    for s, e, line in line_spans(text):
        words = re.findall(r"[a-zà-ÿ]+", line.lower())
        fr, en = sum(w in fr_words for w in words), sum(w in en_words for w in words)
        if fr >= 3 and fr > en:
            out.append((s, e))
    return out

def overlaps(a, b, ranges):
    return any(a < e and s < b for s, e in ranges)

# ---- Emails: leave out automated notifications and emails without zones ----
zones = pd.read_parquet(zones_path, columns=['msgid2', 'start', 'end', 'text', 'zone'])
zones['zone'] = zones['zone'].fillna('empty')
zones['msgid2'] = zones['msgid2'].astype(str)
is_auto = pool['from'].astype(str).str.contains('|'.join(automated_senders), case=False, regex=True)
has_zones = pool['msgid2'].astype(str).isin(zones['msgid2'])
print("Automated emails left out, by sender:")
show(pool.loc[is_auto, 'from'].value_counts().rename('emails').to_frame())
print(f"Emails without zones, left out: {(~is_auto & ~has_zones).sum():,}")
pool9 = pool[~is_auto & has_zones]

# Line starts, ends and zones of each email, in order
zone_lines = {m: (g['start'].to_numpy(), g['end'].to_numpy(), g['zone'].to_numpy(), g['text'].astype(str).to_numpy())
              for m, g in zones.sort_values(['msgid2', 'start']).groupby('msgid2')}

def tag_zone(starts, ends, zs, a):
    """Zone of the line the tag starts on."""
    i = np.searchsorted(starts, a, side='right') - 1
    return zs[i] if i >= 0 and a <= ends[i] else 'none'

# ---- Parse and label every tag ----

rows, mismatched, checked = [], [], 0
for msgid2, body, doc in zip(pool9['msgid2'], pool9['body'].astype(str),
                             nlp.pipe(pool9['body'].astype(str), batch_size=50)):
    starts, ends, zs, texts = zone_lines[str(msgid2)]
    # Check that the zone file's lines match this body
    for s, e, t in zip(starts, ends, texts):
        checked += 1
        if body[s:e].strip() != t.strip():
            mismatched.append(dict(msgid2=msgid2, start=s, zone_file=t, body=body[s:e]))
    fr = french_lines(body)
    names = [(m.start(), m.end()) for p in name_patterns for m in re.finditer(p, body, re.I)]
    formulas = {k: [(m.start(), m.end()) for m in re.finditer(p, body, re.I)]
                for k, (p, cats) in formula_rules.items()}
    for e in doc.ents:
        a, b = e.start_char, e.end_char
        rules = [k for k, (f, cats) in token_rules.items() if (cats is None or e.label_ in cats) and f(e)]
        rules += [k for k, r in formulas.items() if e.label_ in formula_rules[k][1] and overlaps(a, b, r)]
        if overlaps(a, b, names):
            rules.append('name')
        if overlaps(a, b, fr):
            rules.append('french_line')
        rows.append(dict(msgid2=msgid2, tag=e.label_, phrase=' '.join(e.text.split()),
                         start=a, end=b, tok_start=e.start, tok_end=e.end,
                         zone=tag_zone(starts, ends, zs, a), rules=';'.join(rules),
                         left=' '.join(body[max(0, a - 60):a].split()),
                         right=' '.join(body[b:b + 60].split())))

tags9 = pd.DataFrame(rows)
how9 = tags9[tags9['tag'].isin(how_cats)].copy()

# ---- Check: zone file lines against email bodies ----
print(f"\nEmails: {len(pool9):,} | HOW tags: {len(how9):,}")
print(f"Zone-file lines that do not match the email body: {len(mismatched):,} of {checked:,}")
if mismatched:
    show(pd.DataFrame(mismatched).head(10))

# ---- Where HOW tags fall, per category (% of the category's tags) ----
zone_tab = pd.crosstab(how9['tag'], how9['zone'])
zone_tab = zone_tab.div(zone_tab.sum(axis=1), axis=0).mul(100)
zone_tab.insert(0, 'tags', how9['tag'].value_counts())
print("\nWhere HOW tags fall (% of each category's tags):")
show(zone_tab.sort_values('tags', ascending=False).round(1))

# ---- Rule matches (authored text only) ----
auth = how9[how9['zone'] == 'authored']
rule_rows = auth.assign(rule=auth['rules'].str.split(';')).explode('rule')
rule_rows = rule_rows[rule_rows['rule'] != '']
rule_tab = (rule_rows.groupby('rule')
            .agg(tags=('tag', 'size'),
                 categories=('tag', lambda s: '; '.join(f"{k} {v}" for k, v in s.value_counts().head(4).items()))))
rule_tab.insert(0, 'kind', ['fixed' if r in fixed_rules else 'checked' for r in rule_tab.index])
print("\nRule matches in authored text (fixed = applied without coding; checked = coded first):")
show(rule_tab.sort_values(['kind', 'tags'], ascending=[True, False]))

# Authored HOW tags per category matched by any rule
matched = auth.assign(any_rule=auth['rules'] != '')
match_tab = (matched.groupby('tag')['any_rule'].agg(['size', 'sum'])
             .rename(columns={'size': 'authored_tags', 'sum': 'matched_by_rule'}))
match_tab['pct_matched'] = 100 * match_tab['matched_by_rule'] / match_tab['authored_tags']
print("\nAuthored HOW tags matched by any rule, per category:")
show(match_tab.sort_values('authored_tags', ascending=False).round(1))

# ---- Tags that run together (issues spanning more than one tag) ----
# A run is a sequence of tags in authored text with no untagged tokens between them
tags9 = tags9.sort_values(['msgid2', 'tok_start']).reset_index(drop=True)
new_run = ((tags9['msgid2'] != tags9['msgid2'].shift())
           | (tags9['tok_start'] != tags9['tok_end'].shift())
           | (tags9['zone'] != 'authored'))
tags9['run'] = new_run.cumsum()
in_text = tags9[tags9['zone'] == 'authored']

# Candidate names: runs of two or more capitalized tags, at least one of them HOW
cap = in_text['phrase'].str[:1].str.isupper()
name_runs = (in_text[cap].groupby('run')
             .agg(phrase=('phrase', ' '.join), tags=('tag', ' + '.join), n_tags=('tag', 'size'),
                  has_how=('tag', lambda t: t.isin(how_cats).any()),
                  msgid2=('msgid2', 'first'), left=('left', 'first'), right=('right', 'last')))
name_runs = name_runs[(name_runs['n_tags'] > 1) & name_runs['has_how']]
print("\nCandidate names (capitalized runs of 2+ tags), with one example context each:")
name_tab = (name_runs.groupby(['phrase', 'tags'])
            .agg(n=('msgid2', 'size'), msgid2=('msgid2', 'first'),
                 left=('left', 'first'), right=('right', 'first'))
            .reset_index().sort_values('n', ascending=False).head(30))
show(name_tab)

# Candidate formulas: adjacent pairs of tags from different categories, at least one of them HOW
nxt = in_text.shift(-1)
pairs = in_text[(nxt['run'] == in_text['run']) & (nxt['tag'] != in_text['tag'])
                & (in_text['tag'].isin(how_cats) | nxt['tag'].isin(how_cats))]
pairs = pd.DataFrame({
    'phrase': (pairs['phrase'] + ' ' + nxt.loc[pairs.index, 'phrase']).str.lower(),
    'tags': pairs['tag'] + ' + ' + nxt.loc[pairs.index, 'tag']})
print("\nCandidate formulas (adjacent tags from different categories):")
show(pairs.groupby(['phrase', 'tags']).size().rename('n').reset_index()
        .sort_values('n', ascending=False).head(30))

# ---- Coding sample for manual review (stratified) ----
# Strata: each checked rule (up to 30 tags), each noun of domain_noun (up to 10 tags),
# and each HOW category among tags that match no rule (up to 20 tags).
# stratum_size is the number of authored tags in the stratum, for weighting results back in block 13.
n_per_rule, n_per_noun, n_per_category = 30, 10, 20
draws = []
for r, g in rule_rows[~rule_rows['rule'].isin(fixed_rules | {'domain_noun'})].groupby('rule'):
    draws.append(g.sample(min(len(g), n_per_rule), random_state=42)
                 .assign(sample=f"rule:{r}", stratum=f"rule:{r}", stratum_size=len(g)))
nouns = rule_rows[rule_rows['rule'] == 'domain_noun'].assign(noun=lambda d: d['phrase'].str.lower())
for n, g in nouns.groupby('noun'):
    draws.append(g.sample(min(len(g), n_per_noun), random_state=42)
                 .assign(sample='rule:domain_noun', stratum=f"rule:domain_noun:{n}", stratum_size=len(g)))
for c, g in auth[auth['rules'] == ''].groupby('tag'):
    draws.append(g.sample(min(len(g), n_per_category), random_state=42)
                 .assign(sample='no_rule', stratum=f"no_rule:{c}", stratum_size=len(g)))
coding = pd.concat(draws)
coding = coding[~coding.index.duplicated(keep='first')]   # a tag matched by two rules appears once
coding = coding[['sample', 'stratum', 'stratum_size', 'msgid2', 'tag', 'phrase', 'left', 'right', 'rules']]
coding.insert(0, 'id', range(1, len(coding) + 1))
coding['coder1'] = ''
coding['coder2'] = ''
coding['notes'] = ''
coding_path = os.path.join(os.path.dirname(out_path), 'docuscope_coding_sample.csv')
coding.to_csv(coding_path, index=False)
print("\nCoding sample by sample type:")
show(coding.groupby('sample').agg(strata=('stratum', 'nunique'), tags=('id', 'size')))
print(f"Coding sample: {len(coding):,} tags saved to: {coding_path}")

# Shown on each card: what the category means (DocuScope Full Dictionary documentation,
# https://browndw.github.io/docuscope-docs/tagsets/docuscope-fd-tags.html) and why the card is there
category_descriptions = {
    'Interactive': ("Addresses the reader directly: second person, questions, requests.", "can you; thank you for; please see"),
    'Facilitate': ("Guides or helps the reader through tasks and actions.", "let me; worth a try; I would suggest"),
    'MetadiscourseInteractive': ("Reader-facing markers that engage the audience in the exchange.", "I agree; let's talk; by the way"),
    'Inquiry': ("Points to questions, investigation or searching.", "find out; wondering if"),
    'ConfidenceHigh': ("Presents a claim as certain.", "most likely; ensure that; obviously"),
    'ConfidenceHedged': ("Presents a claim as uncertain or tentative.", "maybe; it seems that; tends to get"),
    'ForceStressed': ("Emphatic wording: intensifiers, superlatives, insistence.", "really good; necessary"),
    'Uncertainty': ("Signals that the writer's level of confidence is unknown.", "kind of; I have no idea; for some reason"),
    'Contingent': ("Conditions that depend on circumstances in the world.", "subject to; if possible; hypothetically"),
    'Responsibility': ("Duties, requirements and obligations.", "supposed to; requirements; obligations"),
    'Positive': ("Positive actions, emotions, relations and values.", "thanks; approval; agreement"),
    'Negative': ("Negative actions, emotions, relations and values.", "does not; sorry for; problems"),
    'Future': ("Future actions, states or desires.", "will be; hope to; expected changes"),
    'Updates': ("Signals new information for a reader who is seeking it.", "already; a new; now that"),
    'Narrative': ("Events, people and descriptions extending in time.", "today; tomorrow; this weekend"),
    'Strategic': ("Plans, goals, strategies and seeking advantage.", "plan; trying to; strategy"),
    'FirstPerson': ("References to the writer in the first person.", "I; we have been; as soon as I"),
    'MetadiscourseCohesive': ("Markers that move through the text and link its parts.", "or; but; also"),
    'Reasoning': ("Inferences about cause, consequence, concession or conclusion.", "because; therefore; even if"),
}
rule_descriptions = {
    'domain_noun': "A noun from the data sharing domain (agreement, consent, cohort, research, results and others). "
                   "Flagged because DocuScope may read a document or concept as evaluation or stance.",
    'polite_would': "'would' or 'you would' tagged Narrative. Flagged because polite requests "
                    "('if you would please') describe no event in time.",
    'let_know': "'know' in 'let me/us know' tagged ConfidenceHigh. Flagged because the phrase is a request, "
                "not a claim of certainty.",
    'dont_hesitate': "A word in 'don't hesitate to' tagged Uncertainty, Negative or ForceStressed. Flagged because "
                     "the formula invites contact and expresses no doubt or negativity.",
    'negated_modal': "'do', 'does', 'did' or 'can' before 'not' or 'n't', tagged ForceStressed or ConfidenceHedged. "
                     "Flagged because the word may only be part of a negation.",
    'have_a_day': "A word in 'have a great day/weekend' tagged Strategic or Narrative. Flagged because the formula "
                  "is a farewell, not a plan or an event.",
    'on_my_end': "'end' in 'on my/our/your end' tagged Narrative. Flagged because 'end' means 'side' here, "
                 "not a point in time.",
    'best_of': "'the best of' in 'to the best of your knowledge' tagged Positive. Flagged because the formula "
               "qualifies an answer and evaluates nothing.",
    'date_month': "Fixed rule: a month name next to a day number, tagged outside Narrative (mostly 'May' as hedging).",
    'time_am_pm': "Fixed rule: 'am' after a clock time, tagged FirstPerson.",
    'name': "Fixed rule: part of a multiword name such as COVID-19 Immunity Task Force.",
    'french_line': "Fixed rule: a line of French text, tagged by an English-only model.",
    'terms_cond': "Fixed rule: 'conditions' in 'terms and conditions', tagged Contingent.",
    'no_rule': "No rule matched. A random tag from authored text in this category, used to estimate how often "
               "DocuScope is wrong where no rule applies.",
}

# ---- Coding tool: one HTML page with the sample embedded; each coder exports their own CSV ----
coding_html = r'''<!doctype html>
<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>DocuScope coding</title>
<style>
:root { --bg:#f7f7f5; --card:#fff; --ink:#1f2328; --muted:#6b7075; --line:#e2e2de; --mark:#ffe08a;
        --ok:#2f8f5b; --bad:#c2453d; --unsure:#b7871a; --accent:#4b5fc4; }
@media (prefers-color-scheme: dark) { :root { --bg:#17181a; --card:#212327; --ink:#e8e8e6; --muted:#9a9ea3;
        --line:#33363b; --mark:#7a5f12; --ok:#4fbf83; --bad:#e46a62; --unsure:#e0b04a; --accent:#8b9cf0; } }
* { box-sizing:border-box; }
body { margin:0; background:var(--bg); color:var(--ink); font:16px/1.5 system-ui, -apple-system, sans-serif; }
header { display:flex; flex-wrap:wrap; gap:10px; align-items:center; padding:12px 16px; border-bottom:1px solid var(--line); background:var(--card); position:sticky; top:0; }
header h1 { font-size:16px; margin:0 8px 0 0; }
select, input[type=text], button { font:inherit; font-size:14px; padding:5px 9px; border:1px solid var(--line); border-radius:6px; background:var(--bg); color:var(--ink); }
button { cursor:pointer; }
button.primary { background:var(--accent); color:#fff; border-color:var(--accent); }
.spacer { flex:1; }
.progress { height:4px; background:var(--line); }
.progress div { height:100%; background:var(--accent); width:0; transition:width .2s; }
main { max-width:860px; margin:24px auto; padding:0 16px; }
.card { background:var(--card); border:1px solid var(--line); border-radius:10px; padding:22px; }
.meta { display:flex; flex-wrap:wrap; gap:8px; color:var(--muted); font-size:13px; margin-bottom:14px; }
.pill { border:1px solid var(--line); border-radius:999px; padding:1px 9px; }
.tag { font-size:26px; font-weight:650; margin:0 0 4px; }
.phrase { color:var(--muted); margin-bottom:12px; }
.def { font-size:15px; margin:0 0 4px; }
.def .eg { color:var(--muted); }
.why { font-size:14px; border-left:3px solid var(--accent); padding:6px 12px; margin:10px 0 16px; background:var(--bg); border-radius:0 6px 6px 0; }
.why b { font-weight:600; }
.why div + div { margin-top:4px; }
.context { font-size:19px; line-height:1.7; padding:14px 16px; background:var(--bg); border-radius:8px; }
.context mark { background:var(--mark); color:inherit; padding:1px 3px; border-radius:3px; font-weight:600; }
.codes { display:grid; grid-template-columns:repeat(3,1fr); gap:10px; margin-top:18px; }
.codes button { padding:12px; font-size:15px; border-width:2px; }
.codes button[data-code=correct].on { background:var(--ok); border-color:var(--ok); color:#fff; }
.codes button[data-code=incorrect].on { background:var(--bad); border-color:var(--bad); color:#fff; }
.codes button[data-code=unsure].on { background:var(--unsure); border-color:var(--unsure); color:#fff; }
.note { width:100%; margin-top:12px; }
.nav { display:flex; justify-content:space-between; align-items:center; margin-top:14px; color:var(--muted); font-size:14px; }
.keys { color:var(--muted); font-size:13px; margin-top:16px; text-align:center; }
kbd { border:1px solid var(--line); border-bottom-width:2px; border-radius:4px; padding:0 5px; font-size:12px; background:var(--card); }
.empty { text-align:center; color:var(--muted); padding:40px; }
#about { max-width:860px; margin:20px auto 0; padding:0 16px; }
#about .card { font-size:15px; }
#about h2 { font-size:18px; margin:0 0 8px; }
#about h3 { font-size:15px; margin:18px 0 4px; }
#about p, #about li { margin:4px 0; }
#about table { border-collapse:collapse; font-size:14px; margin-top:6px; }
#about td { border-top:1px solid var(--line); padding:4px 14px 4px 0; vertical-align:top; }
#about[hidden] { display:none; }
@media (max-width:560px) { .codes { grid-template-columns:1fr; } .tag { font-size:22px; } .context { font-size:17px; } }
</style></head>
<body>
<header>
  <h1>DocuScope coding</h1>
  <button id="aboutBtn" aria-expanded="false">About</button>
  <label>Coder <select id="coder"><option value="">choose…</option><option>coder1</option><option>coder2</option></select></label>
  <label>Show <select id="filter"></select></label>
  <label><input type="checkbox" id="uncoded"> uncoded only</label>
  <span class="spacer"></span>
  <span id="count"></span>
  <button id="import">Import</button><input type="file" id="file" accept=".csv" hidden>
  <button id="export" class="primary">Export CSV</button>
</header>
<div class="progress"><div id="bar"></div></div>
<section id="about" hidden><div class="card">
  <h2>About this page</h2>
  <p>This page is the manual check in block 12 of <code>citf_emails_process_mz.ipynb</code>. It shows a stratified sample of HOW tags that DocuScope assigned to authored text in CITF study emails, one at a time, so that two coders can judge whether each tag is accurate.</p>
  <h3>Why we code these tags</h3>
  <p>The coding does two jobs. It tests the error rules in block 12: each checked rule claims that tags like the ones it matches are DocuScope errors, and its sample shows whether that claim holds before block 13 relabels every match in the corpus. It also estimates how often DocuScope is wrong where no rule applies, for each HOW category, which shows which categories are reliable enough to use.</p>
  <p>The codes may later train a model that predicts whether any DocuScope tag is accurate in its context. Whether that is worth doing, and for which categories, depends on the error rates this sample reveals.</p>
  <h3>What a card shows</h3>
  <p>The DocuScope category with a short description and examples, the tagged phrase, why the tag is in the sample (the rule it matched and what that rule claims, or "No rule"), and about 60 characters of text on each side with the phrase highlighted. The pills give the stratum, the number of authored tags in that stratum, and the <code>msgid2</code>. Category descriptions paraphrase the <a href="https://browndw.github.io/docuscope-docs/tagsets/docuscope-fd-tags.html">DocuScope Full Dictionary tag documentation</a>, the tagset of the <code>en_docusco_spacy</code> model.</p>
  <h3>How to code</h3>
  <ul>
    <li><b>correct</b>: the category's meaning fits what the phrase does in that sentence.</li>
    <li><b>incorrect</b>: it does not fit (for example "know" as ConfidenceHigh in "please let me know").</li>
    <li><b>unsure</b>: the context is too short or the case is genuinely ambiguous. Add a note.</li>
  </ul>
  <p>Code alone and without seeing the other coder's file, so that agreement between coders can be measured.</p>
  <h3>How the sample is drawn</h3>
  <p>The sample is stratified, with random draws inside each stratum (<code>random_state=<span id="seed"></span></code>):</p>
  <ul>
    <li><code>rule:&lt;name&gt;</code>: up to <span id="nPerRule"></span> tags matched by each checked rule.</li>
    <li><code>rule:domain_noun:&lt;noun&gt;</code>: up to <span id="nPerNoun"></span> tags for each domain noun, since different nouns may behave differently.</li>
    <li><code>no_rule:&lt;category&gt;</code>: up to <span id="nPerCat"></span> tags per HOW category among tags that match no rule.</li>
  </ul>
  <p>Rules whose matches are treated as errors without coding (<span id="fixedRules"></span>) are applied in block 13. A tag matched by two rules appears once. Each card records its stratum size, so block 13 can weight the coded results back to the whole corpus.</p>
  <table id="sampleTable"></table>
  <h3>Saving and exporting</h3>
  <p>Codes save automatically in this browser, separately for each coder. <b>Export CSV</b> downloads <code>docuscope_coding_coder1.csv</code> or <code>docuscope_coding_coder2.csv</code> with one row per tag: <code>id</code>, sample, stratum, stratum size, <code>msgid2</code>, category, phrase, rules, the code and the note. Save that file in <code>docuscope-assessment/data</code> for block 13. <b>Import</b> reloads an exported file, to resume on another computer or after clearing the browser.</p>
  <h3>Data</h3>
  <p>The page contains excerpts of CITF emails. It works offline and sends nothing anywhere. Keep it and the exported files on storage approved for the study data.</p>
  <p id="generated"></p>
</div></section>
<main id="main"></main>
<p class="keys"><kbd>1</kbd> correct &nbsp; <kbd>2</kbd> incorrect &nbsp; <kbd>3</kbd> unsure &nbsp; <kbd>←</kbd> <kbd>→</kbd> move &nbsp; <kbd>N</kbd> note &nbsp; <kbd>Esc</kbd> leave note</p>
<script id="data" type="application/json">__DATA__</script>
<script id="meta" type="application/json">__META__</script>
<script>
const rows = JSON.parse(document.getElementById('data').textContent);
const meta = JSON.parse(document.getElementById('meta').textContent);
const $ = id => document.getElementById(id);

$('aboutBtn').onclick = () => {
  const open = $('about').hidden;
  $('about').hidden = !open;
  $('aboutBtn').setAttribute('aria-expanded', open);
  try { localStorage.setItem('docuscope-coding-about', open ? '1' : '0'); } catch (e) {}
};
try { if (localStorage.getItem('docuscope-coding-about') !== '0') { $('about').hidden = false; $('aboutBtn').setAttribute('aria-expanded', true); } } catch (e) { $('about').hidden = false; }
$('nPerRule').textContent = meta.n_per_rule;
$('nPerNoun').textContent = meta.n_per_noun;
$('nPerCat').textContent = meta.n_per_category;
$('seed').textContent = meta.random_state;
$('fixedRules').textContent = meta.fixed_rules.join(', ');
$('generated').textContent = `Generated ${meta.generated} from ${meta.emails} emails (${meta.how_tags} HOW tags, ${meta.authored_how_tags} in authored text).`;
let coder = '', codes = {}, view = [], pos = 0;

function storeKey() { return 'docuscope-coding-' + coder; }
function load() { try { codes = JSON.parse(localStorage.getItem(storeKey())) || {}; } catch (e) { codes = {}; } }
function save() { try { localStorage.setItem(storeKey(), JSON.stringify(codes)); } catch (e) {} }
function esc(s) { return String(s ?? '').replace(/[&<>"]/g, c => ({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;'}[c])); }

const samples = [...new Set(rows.map(r => r.sample))];
$('sampleTable').innerHTML = samples.map(s => `<tr><td><code>${esc(s)}</code></td><td>${rows.filter(r => r.sample === s).length === 1 ? '1 tag' : rows.filter(r => r.sample === s).length + ' tags'}</td></tr>`).join('');
$('filter').innerHTML = '<option value="">all samples</option>' +
  samples.map(s => `<option>${esc(s)}</option>`).join('');

function rebuild(keepId) {
  const f = $('filter').value, onlyUncoded = $('uncoded').checked;
  view = rows.filter(r => (!f || r.sample === f) && (!onlyUncoded || !(codes[r.id] && codes[r.id].code)));
  pos = Math.max(0, keepId != null ? view.findIndex(r => r.id === keepId) : 0);
  render();
}

function render() {
  const done = rows.filter(r => codes[r.id] && codes[r.id].code).length;
  $('count').textContent = `${done} / ${rows.length} coded`;
  $('bar').style.width = (100 * done / rows.length) + '%';
  if (!coder) { $('main').innerHTML = '<div class="empty">Choose a coder to start.</div>'; return; }
  if (!view.length) { $('main').innerHTML = '<div class="empty">Nothing to show with these filters.</div>'; return; }
  pos = Math.min(Math.max(pos, 0), view.length - 1);
  const r = view[pos], c = codes[r.id] || {};
  const cat = meta.categories[r.tag];
  const ruleList = r.rules ? r.rules.split(';') : [];
  const why = (ruleList.length ? ruleList : ['no_rule'])
    .map(k => `<div><b>${k === 'no_rule' ? 'No rule' : 'Rule ' + esc(k)}</b>: ${esc(meta.rule_descriptions[k] || '')}</div>`).join('');
  $('main').innerHTML = `
    <div class="card">
      <div class="meta"><span class="pill">${esc(r.stratum)}</span><span class="pill">${Number(r.stratum_size).toLocaleString()} tags in stratum</span><span class="pill">msgid2 ${esc(r.msgid2)}</span></div>
      <div class="tag">${esc(r.tag)}</div>
      ${cat ? `<div class="def">${esc(cat.desc)} <span class="eg">e.g. ${cat.eg.split('; ').map(x => '“' + esc(x) + '”').join(', ')}</span></div>` : ''}
      <div class="phrase">tagged phrase: “${esc(r.phrase)}”</div>
      <div class="why">${why}</div>
      <div class="context">…${esc(r.left)} <mark>${esc(r.phrase)}</mark> ${esc(r.right)}…</div>
      <div class="codes">
        ${['correct','incorrect','unsure'].map((k,i) =>
          `<button data-code="${k}" class="${c.code===k?'on':''}">${i+1} · ${k}</button>`).join('')}
      </div>
      <input class="note" id="note" type="text" placeholder="Note (optional)" value="${esc(c.note)}">
      <div class="nav"><button id="prev">← previous</button><span>${pos+1} of ${view.length}</span><button id="next">next →</button></div>
    </div>`;
  document.querySelectorAll('.codes button').forEach(b => b.onclick = () => setCode(b.dataset.code));
  $('prev').onclick = () => { pos--; render(); };
  $('next').onclick = () => { pos++; render(); };
  $('note').oninput = e => { const id = view[pos].id; codes[id] = {...(codes[id]||{}), note: e.target.value}; save(); };
}

function setCode(k) {
  const r = view[pos];
  codes[r.id] = {...(codes[r.id]||{}), code: k};
  save();
  if ($('uncoded').checked) { view.splice(pos, 1); } else { pos++; }
  render();
}

document.addEventListener('keydown', e => {
  if (!coder || !view.length) return;
  if (e.target.id === 'note') { if (e.key === 'Escape' || e.key === 'Enter') e.target.blur(); return; }
  if (e.target.tagName === 'SELECT' || e.target.tagName === 'INPUT') return;
  if (e.key === '1') setCode('correct');
  else if (e.key === '2') setCode('incorrect');
  else if (e.key === '3') setCode('unsure');
  else if (e.key === 'ArrowRight') { pos++; render(); }
  else if (e.key === 'ArrowLeft') { pos--; render(); }
  else if (e.key.toLowerCase() === 'n') { e.preventDefault(); $('note').focus(); }
});

$('coder').onchange = e => { coder = e.target.value; load(); rebuild(); };
$('filter').onchange = () => rebuild();
$('uncoded').onchange = () => rebuild();

function csvCell(v) { const s = String(v ?? ''); return /[",\n]/.test(s) ? '"' + s.replace(/"/g, '""') + '"' : s; }
$('export').onclick = () => {
  if (!coder) { alert('Choose a coder first.'); return; }
  const cols = ['id','sample','stratum','stratum_size','msgid2','tag','phrase','rules'];
  const lines = [[...cols, coder, coder + '_note'].join(',')].concat(rows.map(r =>
    [...cols.map(k => csvCell(r[k])), csvCell((codes[r.id]||{}).code), csvCell((codes[r.id]||{}).note)].join(',')));
  const a = document.createElement('a');
  a.href = URL.createObjectURL(new Blob([lines.join('\n') + '\n'], {type: 'text/csv'}));
  a.download = `docuscope_coding_${coder}.csv`;
  a.click();
};

function parseCSV(text) {
  const out = []; let row = [], cell = '', q = false;
  for (let i = 0; i < text.length; i++) {
    const ch = text[i];
    if (q) { if (ch === '"' && text[i+1] === '"') { cell += '"'; i++; } else if (ch === '"') q = false; else cell += ch; }
    else if (ch === '"') q = true;
    else if (ch === ',') { row.push(cell); cell = ''; }
    else if (ch === '\n') { row.push(cell); out.push(row); row = []; cell = ''; }
    else if (ch !== '\r') cell += ch;
  }
  if (cell || row.length) { row.push(cell); out.push(row); }
  return out;
}
$('import').onclick = () => { if (!coder) { alert('Choose a coder first.'); return; } $('file').click(); };
$('file').onchange = async e => {
  const f = e.target.files[0]; if (!f) return;
  const [head, ...body] = parseCSV(await f.text());
  const iId = head.indexOf('id'), iCode = head.indexOf(coder), iNote = head.indexOf(coder + '_note');
  if (iId < 0 || iCode < 0) { alert(`That file has no ${coder} column.`); return; }
  body.forEach(r => { if (r[iCode] || r[iNote]) codes[r[iId]] = {code: r[iCode] || undefined, note: iNote >= 0 ? r[iNote] : ''}; });
  save(); rebuild(); e.target.value = '';
};

render();
</script>
</body></html>
'''
tool_rows = coding[['id', 'sample', 'stratum', 'stratum_size', 'msgid2', 'tag', 'phrase',
                    'left', 'right', 'rules']].astype(str).to_dict('records')
tool_json = json.dumps(tool_rows, ensure_ascii=False).replace('</', '<\\/')
tool_meta = dict(generated=pd.Timestamp.now().strftime('%Y-%m-%d %H:%M'), emails=f"{len(pool9):,}",
                 how_tags=f"{len(how9):,}", authored_how_tags=f"{len(auth):,}",
                 n_per_rule=n_per_rule, n_per_noun=n_per_noun, n_per_category=n_per_category,
                 random_state=42, fixed_rules=sorted(fixed_rules),
                 categories={k: {'desc': d, 'eg': eg} for k, (d, eg) in category_descriptions.items()},
                 rule_descriptions=rule_descriptions)
tool_path = os.path.join(os.path.dirname(out_path), 'docuscope_coding_tool.html')
with open(tool_path, 'w', encoding='utf-8') as f:
    f.write(coding_html.replace('__DATA__', tool_json).replace('__META__', json.dumps(tool_meta)))
print(f"Coding tool saved to: {tool_path}")